## One Quad

In [ ]:
from sentence_transformers import SentenceTransformer, util


In [ ]:

def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_score(actual_quad, predicted_quad):
    scores = []
    for actual, predicted in zip(actual_quad, predicted_quad):
        sim = compute_similarity(actual, predicted)
        scores.append(sim)
    return sum(scores) / len(scores) *100



In [ ]:
# Example
actual = ("restaurant service", "good", "dining", "positive")
predicted = ("dining service", "great", "food", "positive")

score = quadruple_score(actual, predicted)
print(f"Quadruple Match Score: {score:.2f}")

Quadruple Match Score: 83.71


In [ ]:
# Pick the model by uncommenting the model assignment statement

#         Model 1: Embedding-based Semantic Similarity (using SentenceTransformer model).ipynb
#         The technique used here is embedding-based semantic similarity, where x and y are encoded into dense vectors using a sentence embedding model, and their similarity is measured by cosine similarity. An exact match shortcut is applied first if the two strings are identical after normalization.
# model = SentenceTransformer('all-MiniLM-L6-v2')
#         Model 2: Instruction-Tuned or Supervised Sentence Embedding Models.ipynb
# model = SentenceTransformer("intfloat/e5-base")  # Drop-in replacement
#         Model 3: GTR / T5-Based Models
#
model = SentenceTransformer("sentence-transformers/gtr-t5-base")



## List of Quad

In [ ]:
def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0  # Return 0 if either list is empty to avoid error and meaningless comparison
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity(actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads(actual_quads, predicted_quads):
    total_score = 0
    matched = 0

    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1

    if matched == 0:
        return 0.0
    return total_score / matched

Example

In [ ]:
# Actual quadruples (ground truth)
actual_quads = [
    ["room", "clean", "service", "positive"],
    ["location", "great", "property", "positive"],
    ["staff", "rude", "service", "negative"],
]

# Predicted quadruples (model output)
predicted_quads = [
    ["location", "excellent", "hotel", "positive"],
    ["room", "very clean", "hotel service", "positive"],
    ["staff", "impolite", "customer service", "negative"]
]

# Run
score = match_quads(actual_quads, predicted_quads)

print(f"Average Quadruple Matching Score: {score:.2f}%")

Average Quadruple Matching Score: 86.01%


# Handle review IDs

In [ ]:
def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity(actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads(actual_quads, predicted_quads):
    total_score = 0
    matched = 0

    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1

    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(review_dict):
    results = {}
    total_score = 0
    total_reviews = 0

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(actual, predicted)
        results[review_id] = round(score, 2)
        total_score += score
        total_reviews += 1

    # Overall performance across all reviews
    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)
    return results


In [ ]:
review_dict = {
    'review_1': {
        'actual': [['room', 'clean', 'service', 'positive']],
        'predicted': [['location', 'excellent', 'hotel', 'positive']]
    },
    'review_2': {
        'actual': [['location', 'great', 'property', 'positive']],
        'predicted': [['room', 'very clean', 'hotel service', 'positive']]
    },
    'review_3': {
        'actual': [['staff', 'rude', 'service', 'negative']],
        'predicted': [['staff', 'impolite', 'customer service', 'negative']]
    }
}

review_scores = evaluate_reviews(review_dict)
print(review_scores)


{'review_1': 67.17, 'review_2': 64.46, 'review_3': 85.66, 'overall_performance': 72.43}


# Keep a Record of Differences for Qualitative Analysis

In [ ]:
def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity(actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads(actual_quads, predicted_quads):
    total_score = 0
    matched = 0

    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1

    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(review_dict, threshold=75.0):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    # Overall performance across all reviews
    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews



In [ ]:
review_dict = {
    'review_1': {
        'actual': [['room', 'clean', 'service', 'positive']],
        'predicted': [['location', 'excellent', 'hotel', 'positive']]
    },
    'review_2': {
        'actual': [['location', 'great', 'property', 'positive']],
        'predicted': [['room', 'very clean', 'hotel service', 'positive']]
    },
    'review_3': {
        'actual': [['staff', 'rude', 'service', 'negative']],
        'predicted': [['staff', 'impolite', 'customer service', 'negative']]
    }
}

review_scores, flagged_reviews = evaluate_reviews(review_dict)

print("Scores:", review_scores)

if len(flagged_reviews) > 0:  # Use len() function correctly
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
elif len(flagged_reviews) == 0:  # Use len() function correctly
    print("\nNo reviews below threshold.")



Scores: {'review_1': 67.17, 'review_2': 64.46, 'review_3': 85.66, 'overall_performance': 72.43}

Reviews below threshold:
{'review_id': 'review_1', 'actual': [['room', 'clean', 'service', 'positive']], 'predicted': [['location', 'excellent', 'hotel', 'positive']], 'total_score': 67.17}
{'review_id': 'review_2', 'actual': [['location', 'great', 'property', 'positive']], 'predicted': [['room', 'very clean', 'hotel service', 'positive']], 'total_score': 64.46}


# ⚖️ HARD PENALTY
* Refere to notion Project 6 for more information on  hard penalty


In [ ]:
def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    # Classification elements must be exact matches
    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()

    # Hard penalty: If either classification element mismatches, return 0
    if not category_match or not sentiment_match:
        return 0.0

    # Compute semantic similarity for aspect and opinion (index 0 and 2)
    scores = []
    for i in [0, 2]:  # aspect and opinion
        sim = compute_similarity(actual_quad[i], predicted_quad[i])
        scores.append(sim)

    return sum(scores) / len(scores) * 100


def match_quads(actual_quads, predicted_quads):
    total_score = 0
    matched = 0

    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1

    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(review_dict, threshold=75.0):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    # Overall performance across all reviews
    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews



In [ ]:
review_dict = {
    'review_1': {
        'actual': [['room', 'clean', 'service', 'positive']],
        'predicted': [['location', 'excellent', 'hotel', 'positive']]
    },
    'review_2': {
        'actual': [['location', 'great', 'property', 'positive']],
        'predicted': [['room', 'very clean', 'hotel service', 'positive']]
    },
    'review_3': {
        'actual': [['staff', 'rude', 'service', 'negative']],
        'predicted': [['staff', 'impolite', 'customer service', 'negative']]
    }
}

review_scores, flagged_reviews = evaluate_reviews(review_dict)

print("Scores:", review_scores)

if len(flagged_reviews) > 0:  # Use len() function correctly
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
elif len(flagged_reviews) == 0:  # Use len() function correctly
    print("\nNo reviews below threshold.")



Scores: {'review_1': 0.0, 'review_2': 0.0, 'review_3': 0.0, 'overall_performance': 0.0}

Reviews below threshold:
{'review_id': 'review_1', 'actual': [['room', 'clean', 'service', 'positive']], 'predicted': [['location', 'excellent', 'hotel', 'positive']], 'total_score': 0.0}
{'review_id': 'review_2', 'actual': [['location', 'great', 'property', 'positive']], 'predicted': [['room', 'very clean', 'hotel service', 'positive']], 'total_score': 0.0}
{'review_id': 'review_3', 'actual': [['staff', 'rude', 'service', 'negative']], 'predicted': [['staff', 'impolite', 'customer service', 'negative']], 'total_score': 0.0}


# Soft Penalties

In [ ]:
def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()

    # Compute similarity for aspect and opinion only
    scores = []
    for i in [0, 2]:  # aspect and opinion
        sim = compute_similarity(actual_quad[i], predicted_quad[i])
        scores.append(sim)

    score = sum(scores) / len(scores) * 100

    # Apply penalty deductions
    if not category_match:
        score *= 0.75  # 25% off
    if not sentiment_match:
        score *= 0.75  # another 25% off

    return score



def match_quads(actual_quads, predicted_quads):
    total_score = 0
    matched = 0

    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1

    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(review_dict, threshold=75.0):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    # Overall performance across all reviews
    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews



In [ ]:
review_dict = {
    'review_1': {
        'actual': [['room', 'clean', 'service', 'positive']],
        'predicted': [['location', 'excellent', 'hotel', 'positive']]
    },
    'review_2': {
        'actual': [['location', 'great', 'property', 'positive']],
        'predicted': [['room', 'very clean', 'hotel service', 'positive']]
    },
    'review_3': {
        'actual': [['staff', 'rude', 'service', 'negative']],
        'predicted': [['staff', 'impolite', 'customer service', 'negative']]
    }
}

review_scores, flagged_reviews = evaluate_reviews(review_dict)

print("Scores:", review_scores)

if len(flagged_reviews) > 0:  # Use len() function correctly
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
elif len(flagged_reviews) == 0:  # Use len() function correctly
    print("\nNo reviews below threshold.")



Scores: {'review_1': 43.8, 'review_2': 39.04, 'review_3': 66.08, 'overall_performance': 49.64}

Reviews below threshold:
{'review_id': 'review_1', 'actual': [['room', 'clean', 'service', 'positive']], 'predicted': [['location', 'excellent', 'hotel', 'positive']], 'total_score': 43.8}
{'review_id': 'review_2', 'actual': [['location', 'great', 'property', 'positive']], 'predicted': [['room', 'very clean', 'hotel service', 'positive']], 'total_score': 39.04}
{'review_id': 'review_3', 'actual': [['staff', 'rude', 'service', 'negative']], 'predicted': [['staff', 'impolite', 'customer service', 'negative']], 'total_score': 66.08}


# Menu-Based Evaluation Script


In [ ]:
from sentence_transformers import SentenceTransformer, util

# ========== Models Available ==========
MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}

# ========== Penalty Options ==========
PENALTY_OPTIONS = {
    "1": "Hard",
    "2": "Soft"
}

# ========== Ask User to Select Options ==========
print("Select Model:")
for key, (name, _) in MODEL_OPTIONS.items():
    print(f"{key}. {name}")
model_choice = input("Enter model number: ").strip()

print("\nSelect Penalty Type:")
for key, name in PENALTY_OPTIONS.items():
    print(f"{key}. {name}")
penalty_choice = input("Enter penalty number: ").strip()

# ========== Load Selected Model ==========
model_name = MODEL_OPTIONS[model_choice][1]
model = SentenceTransformer(model_name)
penalty_type = PENALTY_OPTIONS[penalty_choice]

print(f"\n✅ Using model: {model_name}")
print(f"✅ Penalty strategy: {penalty_type}")

# ========== Similarity Functions ==========

def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()

    # Hard penalty: zero out if any mismatch
    if penalty_type == "Hard" and (not category_match or not sentiment_match):
        return 0.0

    scores = []
    for i in [0, 2]:  # aspect and opinion
        sim = compute_similarity(actual_quad[i], predicted_quad[i])
        scores.append(sim)

    score = sum(scores) / len(scores) * 100

    # Soft penalty: reduce score if mismatched
    if penalty_type == "Soft":
        if not category_match:
            score *= 0.75
        if not sentiment_match:
            score *= 0.75

    return score

def match_quads(actual_quads, predicted_quads):
    total_score = 0
    matched = 0

    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1

    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(review_dict, threshold=75.0):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    results["overall_performance"] = round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0
    return results, under_threshold_reviews

# ========== Sample Data ==========
review_dict = {
    'review_1': {
        'actual': [['room', 'service', 'clean', 'positive']],
        'predicted': [['location', 'staff', 'very clean', 'positive']]
    },
    'review_2': {
        'actual': [['location', 'property', 'great', 'positive']],
        'predicted': [['room', 'property', 'very great', 'positive']]
    },
    'review_3': {
        'actual': [['staff', 'service', 'rude', 'negative']],
        'predicted': [['staff', 'customer service', 'impolite', 'negative']]
    }
}

# ========== Run Evaluation ==========
review_scores, flagged_reviews = evaluate_reviews(review_dict)

print("\n📊 Review Scores:")
for review_id, score in review_scores.items():
    print(f"{review_id}: {score}%")

print("\n⚠️ Reviews below threshold:")
for review in flagged_reviews:
    print(review)


Select Model:
1. Embedding-based (MiniLM)
2. Instruction-Tuned (E5)
3. GTR-T5
Enter model number: 1

Select Penalty Type:
1. Hard
2. Soft
Enter penalty number: 2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ Using model: all-MiniLM-L6-v2
✅ Penalty strategy: Soft

📊 Review Scores:
review_1: 45.26%
review_2: 58.62%
review_3: 55.44%
overall_performance: 53.11%

⚠️ Reviews below threshold:
{'review_id': 'review_1', 'actual': [['room', 'service', 'clean', 'positive']], 'predicted': [['location', 'staff', 'very clean', 'positive']], 'total_score': 45.26}
{'review_id': 'review_2', 'actual': [['location', 'property', 'great', 'positive']], 'predicted': [['room', 'property', 'very great', 'positive']], 'total_score': 58.62}
{'review_id': 'review_3', 'actual': [['staff', 'service', 'rude', 'negative']], 'predicted': [['staff', 'customer service', 'impolite', 'negative']], 'total_score': 55.44}


# Next steps:
1. input and output files
2. transfer excel file to datatype accepted by the evaluation method
3. calculate precision, recall, and f1-score for comparsion between traditional evaluation method and proposed evaluation methods


## Input data from file and output to file

In [ ]:
import csv
from sentence_transformers import SentenceTransformer, util
from google.colab import drive
drive.mount('/content/drive')

# ========== Models Available ==========
MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}

# ========== Penalty Options ==========
PENALTY_OPTIONS = {
    "1": "Hard",
    "2": "Soft"
}

# ========== Ask User to Select Options ==========
print("Select Model:")
for key, (name, _) in MODEL_OPTIONS.items():
    print(f"{key}. {name}")
model_choice = input("Enter model number: ").strip()

print("\nSelect Penalty Type:")
for key, name in PENALTY_OPTIONS.items():
    print(f"{key}. {name}")
penalty_choice = input("Enter penalty number: ").strip()

# ========== Load Selected Model ==========
model_name = MODEL_OPTIONS[model_choice][1]
model = SentenceTransformer(model_name)
penalty_type = PENALTY_OPTIONS[penalty_choice]

print(f"\n✅ Using model: {model_name}")
print(f"✅ Penalty strategy: {penalty_type}")

# ========== Load CSV Data ==========
def load_reviews_from_csv(file_path):
    review_dict = {}
    with open(file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        next(reader)  # Skip the header row
        for row in reader:
            review_id = row['Review ID']

            # Gold (expected) quad
            actual_quad = [
                row['Aspect'].strip(),
                row['Category'].strip(),
                row['Opinion'].strip(),
                row['Sentiment'].strip()
            ]

            # Predicted quad
            predicted_quad = [
                row['Aspect.1'].strip(),
                row['Category.1'].strip(),
                row['Opinion.1'].strip(),
                row['Sentiment.1'].strip()
            ]

            # Save into dictionary
            review_dict[review_id] = {
                'actual': [actual_quad],     # Stored as list of quads
                'predicted': [predicted_quad]
            }
    return review_dict

# ========== Similarity Functions ==========

def compute_similarity(x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()

    # Hard penalty: zero out if any mismatch
    if penalty_type == "Hard" and (not category_match or not sentiment_match):
        return 0.0

    scores = []
    for i in [0, 2]:  # aspect and opinion
        sim = compute_similarity(actual_quad[i], predicted_quad[i])
        scores.append(sim)

    score = sum(scores) / len(scores) * 100

    # Soft penalty: reduce score if mismatched
    if penalty_type == "Soft":
        if not category_match:
            score *= 0.75
        if not sentiment_match:
            score *= 0.75

    return score

def match_quads(actual_quads, predicted_quads):
    total_score = 0
    matched = 0

    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1

    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(review_dict, threshold=75.0):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    results["overall_performance"] = round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0
    return results, under_threshold_reviews

import os

# ========== Main Execution ==========

# file_path = input("\n📄 Enter the path to your CSV file: ").strip()
file_path = '/content/drive/MyDrive/Project6/testdataproj6.csv'

# Check if the file path ends with '.csv', if not, append it
if not file_path.lower().endswith('.csv'):
    file_path += '.csv'

review_dict = load_reviews_from_csv(file_path)

# ========== Updated evaluate_reviews function ==========

def evaluate_reviews(review_dict, threshold=75.0):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    n_tp = 0  # true positives
    n_gold = 0  # total actual quads
    n_pred = 0  # total predicted quads

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])

        # Scoring
        score = match_quads(actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        n_gold += len(actual)
        n_pred += len(predicted)

        # True Positives: exact matches
        for pred_quad in predicted:
            if pred_quad in actual:
                n_tp += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    # Calculate FP and FN
    n_fp = n_pred - n_tp  # False positives = predicted but not correct
    n_fn = n_gold - n_tp  # False negatives = missed gold items

    # Calculate overall precision, recall, f1
    precision = n_tp / n_pred if n_pred > 0 else 0
    recall = n_tp / n_gold if n_gold > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results["overall_performance"] = round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0
    results["precision"] = round(precision * 100, 2)  # in percentage
    results["recall"] = round(recall * 100, 2)
    results["f1_score"] = round(f1 * 100, 2)

    return results, under_threshold_reviews, n_tp, n_fp, n_fn

# ========== Run Evaluation ==========

review_scores, flagged_reviews, n_tp, n_fp, n_fn = evaluate_reviews(review_dict)

# ========== Write Report ==========

# Create output file path (same folder as CSV)
output_folder = os.path.dirname(file_path)
output_file = os.path.join(output_folder, 'evaluation_results.txt')

with open(output_file, 'w', encoding='utf-8') as f:
    f.write("\n📊 Review Scores:\n")
    for review_id, score in review_scores.items():
        if review_id not in ['overall_performance', 'precision', 'recall', 'f1_score']:
            f.write(f"{review_id}: {score}%\n")

    f.write("\n⚠️ Reviews below threshold:\n")
    for review in flagged_reviews:
        f.write(f"{review}\n")

    # ====== NEW: Print TP, FP, FN ======
    f.write("\n🔎 Confusion Matrix Statistics:\n")
    f.write(f"True Positives (TP): {n_tp}\n")
    f.write(f"False Positives (FP): {n_fp}\n")
    f.write(f"False Negatives (FN): {n_fn}\n")

    # ====== Then Print Evaluation Metrics ======
    f.write("\n📈 Overall Evaluation Metrics:\n")
    f.write(f"Overall Average Score: {review_scores['overall_performance']}%\n")
    f.write(f"Precision: {review_scores['precision']}%\n")
    f.write(f"Recall: {review_scores['recall']}%\n")
    f.write(f"F1-Score: {review_scores['f1_score']}%\n")

print(f"\n✅ Evaluation results saved in '{output_file}'.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Select Model:
1. Embedding-based (MiniLM)
2. Instruction-Tuned (E5)
3. GTR-T5
Enter model number: 2

Select Penalty Type:
1. Hard
2. Soft
Enter penalty number: 2

✅ Using model: intfloat/e5-base
✅ Penalty strategy: Soft

✅ Evaluation results saved in '/content/drive/MyDrive/Project6/evaluation_results.txt'.


# Next step
* Calculate the time for each evaluation
* The report should list the three models with each panelty


## Test all three models


In [ ]:
import csv
import os
import time

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Models and penalties
MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]
# In Soft penalty mode, apply a 50% reduction to the similarity score for each mismatch in Category or Sentiment, allowing partial credit instead of discarding the prediction.
# If Soft Penalty high (Ex 0.75) the Soft penalty isn't reducing the score enough to change the final rounded percentage, therfor the hard and soft will generate same results
SOFT_PENALTY_FACTOR = 0.50

# Load Reviews
file_path = '/content/drive/MyDrive/Project6/testdataproj6.csv'
if not file_path.lower().endswith('.csv'):
    file_path += '.csv'

def load_reviews_from_csv(file_path):
    review_dict = {}
    with open(file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        next(reader)  # skip header
        for row in reader:
            review_id = row['Review ID']
            actual_quad = [row['Aspect'].strip(), row['Category'].strip(), row['Opinion'].strip(), row['Sentiment'].strip()]
            predicted_quad = [row['Aspect.1'].strip(), row['Category.1'].strip(), row['Opinion.1'].strip(), row['Sentiment.1'].strip()]
            review_dict[review_id] = {'actual': [actual_quad], 'predicted': [predicted_quad]}
    return review_dict

review_dict = load_reviews_from_csv(file_path)

# Similarity and Evaluation

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard" and (not category_match or not sentiment_match):
        return 0.0

    scores = []
    for i in [0, 2]:  # aspect, opinion
        sim = compute_similarity(model, actual_quad[i], predicted_quad[i])
        scores.append(sim)
    score = sum(scores) / len(scores) * 100

    if penalty_type == "Soft":
        if not category_match:
            score *= SOFT_PENALTY_FACTOR
        if not sentiment_match:
            score *= SOFT_PENALTY_FACTOR
    return score

def match_quads(model, penalty_type, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(model, penalty_type, review_dict, threshold=75.0):
    results = {}
    total_score = 0
    total_reviews = 0
    n_tp = 0
    n_gold = 0
    n_pred = 0

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])

        score = match_quads(model, penalty_type, actual, predicted)
        total_score += score
        total_reviews += 1

        n_gold += len(actual)
        n_pred += len(predicted)

        for pred_quad in predicted:
            if pred_quad in actual:
                n_tp += 1

    n_fp = n_pred - n_tp
    n_fn = n_gold - n_tp

    precision = n_tp / n_pred if n_pred > 0 else 0
    recall = n_tp / n_gold if n_gold > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'Average Score': round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0,
        'Precision': round(precision * 100, 2),
        'Recall': round(recall * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'TP': n_tp,
        'FP': n_fp,
        'FN': n_fn
    }

# Run for all models and penalties
output_folder = os.path.dirname(file_path)
output_file = os.path.join(output_folder, 'evaluation_results.txt')

with open(output_file, 'w', encoding='utf-8') as f:
    last_evaluation = None  # to save the last evaluation

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            start_time = time.time()
            evaluation = evaluate_reviews(model, penalty_type, review_dict)
            end_time = time.time()
            elapsed_time = end_time - start_time

            last_evaluation = evaluation  # save latest evaluation

            # Build text
            title = f"\n====== {model_name} | Penalty: {penalty_type} ======\n"
            avg_score_line = f"Average Score: {evaluation['Average Score']}%"
            time_line = f"Elapsed Time: {elapsed_time:.2f} seconds"

            # Save to file
            for line in [title, avg_score_line, time_line]:
                f.write(line + '\n')

            # Print to screen
            print(title.strip())
            print(avg_score_line)
            print(time_line)
            print("-" * 50)

    # After all evaluations finished ✅
    if last_evaluation:
        final_block = [
            "\n====== Overall Final Metrics ======",
            f"Precision: {last_evaluation['Precision']}%",
            f"Recall: {last_evaluation['Recall']}%",
            f"F1-Score: {last_evaluation['F1-Score']}%",
            "",
            f"True Positives (TP): {last_evaluation['TP']}",
            f"False Positives (FP): {last_evaluation['FP']}",
            f"False Negatives (FN): {last_evaluation['FN']}"
        ]

        # Save final metrics to file
        for line in final_block:
            f.write(line + '\n')

        # Print final metrics to screen
        for line in final_block:
            print(line)


print(f"\n✅ All evaluations completed. Results saved in '{output_file}'.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== Embedding-based (MiniLM) | Penalty: Hard ======
Average Score: 81.25%
Elapsed Time: 0.15 seconds
--------------------------------------------------
====== Embedding-based (MiniLM) | Penalty: Soft ======
Average Score: 83.72%
Elapsed Time: 0.18 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Hard ======
Average Score: 88.7%
Elapsed Time: 0.71 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Soft ======
Average Score: 92.41%
Elapsed Time: 0.76 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Hard ======
Average Score: 85.67%
Elapsed Time: 0.88 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Soft ======
Average Score: 88.81%
Elapsed Time: 1.23 seconds
---------------------------------

# Adding the qualitative analysis file

In [ ]:
import csv
import os
import time
import math
import csv

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Models and penalties
MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]
# In Soft penalty mode, apply a 50% reduction to the similarity score for each mismatch in Category or Sentiment, allowing partial credit instead of discarding the prediction.
# If Soft Penalty high (Ex 0.75) the Soft penalty isn't reducing the score enough to change the final rounded percentage, therefore the hard and soft will generate same results
SOFT_PENALTY_FACTOR = 0.50
MIN_QUAL_SAMPLES = 15  # Minimum number of qualitative samples per model
THRESHOLD = 75.0  # minimum score to consider a prediction acceptable
MAX_QUAL_SAMPLES = 50  # optional upper limit

# Load Reviews
# 🔧 Update this path to point to your review CSV file location

file_path = '/content/drive/MyDrive/Project6/testdataproj6.csv'
if not file_path.lower().endswith('.csv'):
    file_path += '.csv'

def load_reviews_from_csv(file_path):
    review_dict = {}
    with open(file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        next(reader)  # skip header
        for row in reader:
            review_id = row['Review ID']
            actual_quad = [row['Aspect'].strip(), row['Category'].strip(), row['Opinion'].strip(), row['Sentiment'].strip()]
            predicted_quad = [row['Aspect.1'].strip(), row['Category.1'].strip(), row['Opinion.1'].strip(), row['Sentiment.1'].strip()]
            review_dict[review_id] = {
                'review': row['Review'],
                'actual': [actual_quad],
                'predicted': [predicted_quad]
            }
    return review_dict

review_dict = load_reviews_from_csv(file_path)

# Similarity and Evaluation

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard" and (not category_match or not sentiment_match):
        return 0.0

    scores = []
    for i in [0, 2]:  # aspect, opinion
        sim = compute_similarity(model, actual_quad[i], predicted_quad[i])
        scores.append(sim)
    score = sum(scores) / len(scores) * 100

    if penalty_type == "Soft":
        if not category_match:
            score *= SOFT_PENALTY_FACTOR
        if not sentiment_match:
            score *= SOFT_PENALTY_FACTOR
    return score

def match_quads(model, penalty_type, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews(model, penalty_type, review_dict, threshold=THRESHOLD):
    results = {}
    total_score = 0
    total_reviews = 0
    n_tp = 0
    n_gold = 0
    n_pred = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(model, penalty_type, actual, predicted)
        total_score += score
        total_reviews += 1

        n_gold += len(actual)
        n_pred += len(predicted)

        for pred_quad in predicted:
            if pred_quad in actual:
                n_tp += 1

        if score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "review_text": quads.get("review", ""),
                "actual": actual,
                "predicted": predicted,
                "total_score": round(score, 2)
            })

    n_fp = n_pred - n_tp
    n_fn = n_gold - n_tp

    precision = n_tp / n_pred if n_pred > 0 else 0
    recall = n_tp / n_gold if n_gold > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'Average Score': round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0,
        'Precision': round(precision * 100, 2),
        'Recall': round(recall * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'TP': n_tp,
        'FP': n_fp,
        'FN': n_fn
    }, under_threshold_reviews

def write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews, output_path):
    under_threshold_reviews.sort(key=lambda x: x['total_score'])
    sample_size = min(max(MIN_QUAL_SAMPLES, math.ceil(len(under_threshold_reviews) * 0.10)), MAX_QUAL_SAMPLES)
    selected_samples = under_threshold_reviews[:sample_size]

    with open(output_path, 'a', encoding='utf-8') as qf:
        qf.write(f"\n====== {model_name} | Penalty: {penalty_type} ======\n")
        qf.write(f"Total under-threshold reviews: {len(under_threshold_reviews)}\n")
        qf.write(f"Showing {len(selected_samples)} lowest scoring examples\n")

        for case in selected_samples:
            qf.write(f"\nReview ID: {case['review_id']}\n")
            qf.write(f"Review Text: {case['review_text']}\n")
            qf.write(f"Score: {case['total_score']}%\n")
            qf.write("Expected Quad(s):\n")
            for quad in case['actual']:
                qf.write(f"  - {quad}\n")
            qf.write("Predicted Quad(s):\n")
            for quad in case['predicted']:
                qf.write(f"  - {quad}\n")

# Run for all models and penalties
output_folder = os.path.dirname(file_path)
output_file = os.path.join(output_folder, 'evaluation_results.txt')
qual_file = os.path.join(output_folder, 'qualitative_analysis.txt')

with open(output_file, 'w', encoding='utf-8') as f:
    last_evaluation = None  # to save the last evaluation

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            start_time = time.time()
            evaluation, under_threshold_reviews = evaluate_reviews(model, penalty_type, review_dict, threshold=THRESHOLD)
            end_time = time.time()
            elapsed_time = end_time - start_time

            last_evaluation = evaluation  # save latest evaluation

            # Build text
            title = f"\n====== {model_name} | Penalty: {penalty_type} ======\n"
            avg_score_line = f"Average Score: {evaluation['Average Score']}%"
            time_line = f"Elapsed Time: {elapsed_time:.2f} seconds"

            # Save to file
            for line in [title, avg_score_line, time_line]:
                f.write(line + '\n')

            # Print to screen
            print(title.strip())
            print(avg_score_line)
            print(time_line)
            print("-" * 50)

            # Write qualitative analysis
            write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews, qual_file)

    # After all evaluations finished ✅
    if last_evaluation:
        final_block = [
            "\n====== Overall Final Metrics ======",
            f"Precision: {last_evaluation['Precision']}%",
            f"Recall: {last_evaluation['Recall']}%",
            f"F1-Score: {last_evaluation['F1-Score']}%",
            "",
            f"True Positives (TP): {last_evaluation['TP']}",
            f"False Positives (FP): {last_evaluation['FP']}",
            f"False Negatives (FN): {last_evaluation['FN']}"
        ]

        # Save final metrics to file
        for line in final_block:
            f.write(line + '\n')

        # Print final metrics to screen
        for line in final_block:
            print(line)

print(f"\n✅ All evaluations completed. Results saved in '{output_file}' and qualitative samples in '{qual_file}'.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== Embedding-based (MiniLM) | Penalty: Hard ======
Average Score: 81.25%
Elapsed Time: 0.18 seconds
--------------------------------------------------
====== Embedding-based (MiniLM) | Penalty: Soft ======
Average Score: 83.72%
Elapsed Time: 0.23 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Hard ======
Average Score: 88.7%
Elapsed Time: 0.59 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Soft ======
Average Score: 92.41%
Elapsed Time: 0.71 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Hard ======
Average Score: 85.67%
Elapsed Time: 0.55 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Soft ======
Average Score: 88.81%
Elapsed Time: 0.70 seconds
---------------------------------

# Export Evaluation Summary to CSV

In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Models and penalties
MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

# Configurations
SOFT_PENALTY_FACTOR = 0.50
MIN_QUAL_SAMPLES = 15
MAX_QUAL_SAMPLES = 50
THRESHOLD = 75.0

# 🔧 Update this path to point to your review CSV file location
file_path = '/content/drive/MyDrive/Project6/testdataproj6.csv'
if not file_path.lower().endswith('.csv'):
    file_path += '.csv'

# Output files
output_folder = os.path.dirname(file_path)
output_file = os.path.join(output_folder, 'evaluation_results.txt')
qual_file = os.path.join(output_folder, 'qualitative_analysis.txt')
summary_csv = os.path.join(output_folder, 'evaluation_summary.csv')

summary_rows = []

def load_reviews_from_csv(file_path):
    review_dict = {}
    with open(file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        next(reader)
        for row in reader:
            review_id = row['Review ID']
            actual_quad = [row['Aspect'].strip(), row['Category'].strip(), row['Opinion'].strip(), row['Sentiment'].strip()]
            predicted_quad = [row['Aspect.1'].strip(), row['Category.1'].strip(), row['Opinion.1'].strip(), row['Sentiment.1'].strip()]
            review_dict[review_id] = {
                'review': row['Review'],
                'actual': [actual_quad],
                'predicted': [predicted_quad]
            }
    return review_dict

review_dict = load_reviews_from_csv(file_path)

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard" and (not category_match or not sentiment_match):
        return 0.0
    scores = [compute_similarity(model, actual_quad[i], predicted_quad[i]) for i in [0, 2]]
    score = sum(scores) / len(scores) * 100
    if penalty_type == "Soft":
        if not category_match:
            score *= SOFT_PENALTY_FACTOR
        if not sentiment_match:
            score *= SOFT_PENALTY_FACTOR
    return score

def match_quads(model, penalty_type, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    return total_score / matched if matched > 0 else 0.0

def evaluate_reviews(model, penalty_type, review_dict, threshold=THRESHOLD):
    total_score = 0
    total_reviews = 0
    n_tp = 0
    n_gold = 0
    n_pred = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(model, penalty_type, actual, predicted)
        total_score += score
        total_reviews += 1
        n_gold += len(actual)
        n_pred += len(predicted)
        for pred_quad in predicted:
            if pred_quad in actual:
                n_tp += 1
        if score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "review_text": quads.get("review", ""),
                "actual": actual,
                "predicted": predicted,
                "total_score": round(score, 2)
            })

    n_fp = n_pred - n_tp
    n_fn = n_gold - n_tp
    precision = n_tp / n_pred if n_pred > 0 else 0
    recall = n_tp / n_gold if n_gold > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'Average Score': round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0,
        'Precision': round(precision * 100, 2),
        'Recall': round(recall * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'TP': n_tp,
        'FP': n_fp,
        'FN': n_fn
    }, under_threshold_reviews

def write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews, output_path):
    under_threshold_reviews.sort(key=lambda x: x['total_score'])
    sample_size = min(max(MIN_QUAL_SAMPLES, math.ceil(len(under_threshold_reviews) * 0.10)), MAX_QUAL_SAMPLES)
    selected_samples = under_threshold_reviews[:sample_size]

    with open(output_path, 'a', encoding='utf-8') as qf:
        qf.write(f"\n====== {model_name} | Penalty: {penalty_type} ======\n")
        qf.write(f"Total under-threshold reviews: {len(under_threshold_reviews)}\n")
        qf.write(f"Showing {len(selected_samples)} lowest scoring examples\n")
        for case in selected_samples:
            qf.write(f"\nReview ID: {case['review_id']}\n")
            qf.write(f"Review Text: {case['review_text']}\n")
            qf.write(f"Score: {case['total_score']}%\n")
            qf.write("Expected Quad(s):\n")
            for quad in case['actual']:
                qf.write(f"  - {quad}\n")
            qf.write("Predicted Quad(s):\n")
            for quad in case['predicted']:
                qf.write(f"  - {quad}\n")

with open(output_file, 'w', encoding='utf-8') as f:
    last_evaluation = None
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            start_time = time.time()
            evaluation, under_threshold_reviews = evaluate_reviews(model, penalty_type, review_dict)
            end_time = time.time()
            elapsed_time = end_time - start_time
            last_evaluation = evaluation

            title = f"\n====== {model_name} | Penalty: {penalty_type} ======\n"
            avg_score_line = f"Average Score: {evaluation['Average Score']}%"
            time_line = f"Elapsed Time: {elapsed_time:.2f} seconds"
            for line in [title, avg_score_line, time_line]:
                f.write(line + '\n')
            print(title.strip())
            print(avg_score_line)
            print(time_line)
            print("-" * 50)

            # Add row for summary CSV
            summary_rows.append([
                model_name,
                penalty_type,
                evaluation['Average Score'],
                evaluation['Precision'],
                evaluation['Recall'],
                evaluation['F1-Score'],
                evaluation['TP'],
                evaluation['FP'],
                evaluation['FN'],
                round(elapsed_time, 2)
            ])

            write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews, qual_file)

    if last_evaluation:
        final_block = [
            "\n====== Overall Final Metrics ======",
            f"Precision: {last_evaluation['Precision']}%",
            f"Recall: {last_evaluation['Recall']}%",
            f"F1-Score: {last_evaluation['F1-Score']}%",
            "",
            f"True Positives (TP): {last_evaluation['TP']}",
            f"False Positives (FP): {last_evaluation['FP']}",
            f"False Negatives (FN): {last_evaluation['FN']}"
        ]
        for line in final_block:
            f.write(line + '\n')
            print(line)

with open(summary_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Model", "Penalty", "Avg Score", "Precision", "Recall", "F1", "TP", "FP", "FN", "Time (s)"])
    for row in summary_rows:
        writer.writerow(row)

print(f"\n✅ All evaluations completed. Results saved in '{output_file}', qualitative samples in '{qual_file}', and summary in '{summary_csv}'.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== Embedding-based (MiniLM) | Penalty: Hard ======
Average Score: 81.25%
Elapsed Time: 0.51 seconds
--------------------------------------------------
====== Embedding-based (MiniLM) | Penalty: Soft ======
Average Score: 83.72%
Elapsed Time: 0.23 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Hard ======
Average Score: 88.7%
Elapsed Time: 0.59 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Soft ======
Average Score: 92.41%
Elapsed Time: 0.71 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Hard ======
Average Score: 85.67%
Elapsed Time: 0.56 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Soft ======
Average Score: 88.81%
Elapsed Time: 0.71 seconds
---------------------------------

# 🧪 Compare Multiple Models on Same Reviews


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Models and penalties
MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

# Configurations
SOFT_PENALTY_FACTOR = 0.50
MIN_QUAL_SAMPLES = 15
MAX_QUAL_SAMPLES = 50
THRESHOLD = 75.0

# 🔧 Update this path to point to your review CSV file location
file_path = '/content/drive/MyDrive/Project6/testdataproj6.csv'
if not file_path.lower().endswith('.csv'):
    file_path += '.csv'

# Output files
output_folder = os.path.dirname(file_path)
output_file = os.path.join(output_folder, 'evaluation_results.txt')
qual_file = os.path.join(output_folder, 'qualitative_analysis.csv')
summary_csv = os.path.join(output_folder, 'evaluation_summary.csv')

summary_rows = []

def load_reviews_from_csv(file_path):
    review_dict = {}
    with open(file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        next(reader)
        for row in reader:
            review_id = row['Review ID']
            actual_quad = [row['Aspect'].strip(), row['Category'].strip(), row['Opinion'].strip(), row['Sentiment'].strip()]
            predicted_quad = [row['Aspect.1'].strip(), row['Category.1'].strip(), row['Opinion.1'].strip(), row['Sentiment.1'].strip()]
            review_dict[review_id] = {
                'review': row['Review'],
                'actual': [actual_quad],
                'predicted': [predicted_quad]
            }
    return review_dict

review_dict = load_reviews_from_csv(file_path)

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard" and (not category_match or not sentiment_match):
        return 0.0
    scores = [compute_similarity(model, actual_quad[i], predicted_quad[i]) for i in [0, 2]]
    score = sum(scores) / len(scores) * 100
    if penalty_type == "Soft":
        if not category_match:
            score *= SOFT_PENALTY_FACTOR
        if not sentiment_match:
            score *= SOFT_PENALTY_FACTOR
    return score

def match_quads(model, penalty_type, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    return total_score / matched if matched > 0 else 0.0

def evaluate_reviews(model, penalty_type, review_dict, threshold=THRESHOLD):
    total_score = 0
    total_reviews = 0
    n_tp = 0
    n_gold = 0
    n_pred = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(model, penalty_type, actual, predicted)
        total_score += score
        total_reviews += 1
        n_gold += len(actual)
        n_pred += len(predicted)
        for pred_quad in predicted:
            if pred_quad in actual:
                n_tp += 1
        if score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "review_text": quads.get("review", ""),
                "actual": actual,
                "predicted": predicted,
                "total_score": round(score, 2)
            })

    n_fp = n_pred - n_tp
    n_fn = n_gold - n_tp
    precision = n_tp / n_pred if n_pred > 0 else 0
    recall = n_tp / n_gold if n_gold > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'Average Score': round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0,
        'Precision': round(precision * 100, 2),
        'Recall': round(recall * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'TP': n_tp,
        'FP': n_fp,
        'FN': n_fn
    }, under_threshold_reviews

qualitative_rows = []

def write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews):
    under_threshold_reviews.sort(key=lambda x: x['total_score'])
    sample_size = min(max(MIN_QUAL_SAMPLES, math.ceil(len(under_threshold_reviews) * 0.10)), MAX_QUAL_SAMPLES)
    selected_samples = under_threshold_reviews[:sample_size]

    for case in selected_samples:
        qualitative_rows.append([
            model_name,
            penalty_type,
            case['review_id'],
            case['review_text'],
            case['total_score'],
            case['actual'],
            case['predicted']
        ])



with open(output_file, 'w', encoding='utf-8') as f:
    last_evaluation = None
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            start_time = time.time()
            evaluation, under_threshold_reviews = evaluate_reviews(model, penalty_type, review_dict)

            end_time = time.time()
            elapsed_time = end_time - start_time
            last_evaluation = evaluation

            title = f"\n====== {model_name} | Penalty: {penalty_type} ======\n"
            avg_score_line = f"Average Score: {evaluation['Average Score']}%"
            time_line = f"Elapsed Time: {elapsed_time:.2f} seconds"
            for line in [title, avg_score_line, time_line]:
                f.write(line + '\n')
            print(title.strip())
            print(avg_score_line)
            print(time_line)
            print("-" * 50)

            # Add row for summary CSV
            summary_rows.append([
                model_name,
                penalty_type,
                evaluation['Average Score'],
                round(elapsed_time, 2)
            ])

            write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews)

    if last_evaluation:
        final_block = ["====== Overall Final Metrics ======",
            f"Precision: {last_evaluation['Precision']}%",
            f"Recall: {last_evaluation['Recall']}%",
            f"F1-Score: {last_evaluation['F1-Score']}%",
            f"True Positives (TP): {last_evaluation['TP']}",
            f"False Positives (FP): {last_evaluation['FP']}",
            f"False Negatives (FN): {last_evaluation['FN']}"
        ]
        for line in final_block:
            f.write(line + '\n')
            print(line)

with open(summary_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Model", "Penalty", "Avg Score", "Time (s)"])
    for row in summary_rows:
        writer.writerow(row)

# Write sorted qualitative rows to CSV
qualitative_rows.sort(key=lambda x: x[2])  # sort by Review ID
with open(qual_file, 'w', newline='', encoding='utf-8') as qf:
    writer = csv.writer(qf)
    writer.writerow(["Model", "Penalty", "Review ID", "Review Text", "Score", "Actual Quad(s)", "Predicted Quad(s)"])
    for row in qualitative_rows:
        writer.writerow(row)

print(f"✅ All evaluations completed. Results saved in '{output_file}', qualitative samples in '{qual_file}', and summary in '{summary_csv}'.")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== Embedding-based (MiniLM) | Penalty: Hard ======
Average Score: 81.25%
Elapsed Time: 0.11 seconds
--------------------------------------------------
====== Embedding-based (MiniLM) | Penalty: Soft ======
Average Score: 83.72%
Elapsed Time: 0.14 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Hard ======
Average Score: 88.7%
Elapsed Time: 0.57 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Soft ======
Average Score: 92.41%
Elapsed Time: 0.73 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Hard ======
Average Score: 85.67%
Elapsed Time: 0.55 seconds
--------------------------------------------------
====== GTR-T5 | Penalty: Soft ======
Average Score: 88.81%
Elapsed Time: 0.71 seconds
---------------------------------

# 📂 Automatically create a timestamped folder for evaluation output
## Folder name format: HHMMDDMMYYYY (e.g., 022830042025 for 02:28 on 30/04/2025)


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


# Models and penalties
MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

# Configurations
SOFT_PENALTY_FACTOR = 0.50
MIN_QUAL_SAMPLES = 15
MAX_QUAL_SAMPLES = 50
THRESHOLD = 75.0

# 🔧 Update these paths to point to your CSV file locations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'

print(f"✅ (debug) Drive mounted, checking file paths:\nHuman File: {human_file_path} -> Exists: {os.path.exists(human_file_path)}\nChatGPT File: {chatgpt_file_path} -> Exists: {os.path.exists(chatgpt_file_path)}")

# Ensure both paths end with '.csv'
if not human_file_path.lower().endswith('.csv'):
    human_file_path += '.csv'
if not chatgpt_file_path.lower().endswith('.csv'):
    chatgpt_file_path += '.csv'


# Output folder named by current time and date
from datetime import datetime
now = datetime.now()
eval_folder_name = now.strftime("%H%M%d%m%Y")
output_folder = os.path.join(os.path.dirname(human_file_path), eval_folder_name)
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, 'evaluation_results.txt')
qual_file = os.path.join(output_folder, 'qualitative_analysis.csv')
summary_csv = os.path.join(output_folder, 'evaluation_summary.csv')

summary_rows = []

import pandas as pd

import pandas as pd

def load_reviews_from_csv(human_file_path, chatgpt_file_path):
    # Load the two CSV files into DataFrames
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    # Convert all string cells to lowercase
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    print(f"✅ (debug) Loaded HumanAnnotations.csv with {len(human_df)} rows and columns: {list(human_df.columns)}")
    print(f"✅ (debug) Loaded ChatGPTAnnotations.csv with {len(chatgpt_df)} rows and columns: {list(chatgpt_df.columns)}")

    review_dict = {}

    # Get all unique Review IDs (from both files combined)
    review_ids = pd.concat([human_df['Review ID'], chatgpt_df['Review ID']]).unique()
    print(f"✅ (debug) Found {len(review_ids)} unique Review IDs")

    # Ensure Review IDs are strings in both DataFrames
    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)


    for review_id in review_ids:
        # Get the review text (from either file – assume it's the same within the same Review ID)
        # We'll try HumanAnnotations first; fallback to ChatGPTAnnotations if needed
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]



        # Handle cases where there may be no matching review text
        if not human_rows.empty:
            review_text = human_rows['Review'].iloc[0]
        elif not chatgpt_rows.empty:
            review_text = chatgpt_rows['Review'].iloc[0]
        else:
            review_text = ""

        # Collect actual quads
        actual_quads = []
        for _, h_row in human_rows.iterrows():
            actual_quad = [
                str(h_row['Aspect']).strip(),
                str(h_row['Category']).strip(),
                str(h_row['Opinion']).strip(),
                str(h_row['Sentiment']).strip()
            ]
            actual_quads.append(actual_quad)

        # Collect predicted quads
        predicted_quads = []
        for _, c_row in chatgpt_rows.iterrows():
            predicted_quad = [
                str(c_row['Aspect']).strip(),
                str(c_row['Category']).strip(),
                str(c_row['Opinion']).strip(),
                str(c_row['Sentiment']).strip()
            ]
            predicted_quads.append(predicted_quad)
        # Add to the dictionary if we have any annotations
        if actual_quads or predicted_quads:
            review_dict[review_id] = {
                'review': review_text,
                'actual': actual_quads,
                'predicted': predicted_quads
            }

    return review_dict


review_dict = load_reviews_from_csv(human_file_path, chatgpt_file_path)

# ✅ (debug) Check that reviews are loaded and print one example
if len(review_dict) > 0:
    sample_key = next(iter(review_dict))
    print(f"✅ (debug) Sample review ID: {sample_key}")
    print(f"✅ (debug) Sample actual quads: {review_dict[sample_key]['actual']}")
    print(f"✅ (debug) Sample predicted quads: {review_dict[sample_key]['predicted']}")

else:
    print("✅ (debug) No reviews found in review_dict. Check your CSV files.")



def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim


def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    # Aspect (embedding sim)
    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])

    # Opinion (embedding sim)
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    # Category (exact match or soft penalty)
    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    # Sentiment (exact match or soft penalty)
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    # Final weighted score (out of 100)
    final_score = (
        aspect_similarity * 25 +
        opinion_similarity * 25 +
        category_score * 25 +
        sentiment_score * 25
    )

    return final_score


def match_quads(model, penalty_type, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    return total_score / matched if matched > 0 else 0.0

def evaluate_reviews(model, penalty_type, review_dict, threshold=THRESHOLD):
    total_score = 0
    total_reviews = 0
    n_tp = 0
    n_gold = 0
    n_pred = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads(model, penalty_type, actual, predicted)
        total_score += score
        total_reviews += 1
        n_gold += len(actual)
        n_pred += len(predicted)
        for pred_quad in predicted:
            if pred_quad in actual:
                n_tp += 1
        if score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "review_text": quads.get("review", ""),
                "actual": actual,
                "predicted": predicted,
                "total_score": round(score, 2)
            })

    n_fp = n_pred - n_tp
    n_fn = n_gold - n_tp
    precision = n_tp / n_pred if n_pred > 0 else 0
    recall = n_tp / n_gold if n_gold > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'Average Score': round(total_score / total_reviews, 2) if total_reviews > 0 else 0.0,
        'Precision': round(precision * 100, 2),
        'Recall': round(recall * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'TP': n_tp,
        'FP': n_fp,
        'FN': n_fn
    }, under_threshold_reviews

qualitative_rows = []

def write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews):
    under_threshold_reviews.sort(key=lambda x: x['total_score'])
    sample_size = min(max(MIN_QUAL_SAMPLES, math.ceil(len(under_threshold_reviews) * 0.10)), MAX_QUAL_SAMPLES)
    selected_samples = under_threshold_reviews[:sample_size]

    for case in selected_samples:
        qualitative_rows.append([
            model_name,
            penalty_type,
            case['review_id'],
            case['review_text'],
            case['total_score'],
            case['actual'],
            case['predicted']
        ])



with open(output_file, 'w', encoding='utf-8') as f:
    last_evaluation = None
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            start_time = time.time()
            evaluation, under_threshold_reviews = evaluate_reviews(model, penalty_type, review_dict)

            end_time = time.time()
            elapsed_time = end_time - start_time
            last_evaluation = evaluation

            title = f"\n====== {model_name} | Penalty: {penalty_type} ======\n"
            avg_score_line = f"Average Score: {evaluation['Average Score']}%"
            time_line = f"Elapsed Time: {elapsed_time:.2f} seconds"
            for line in [title, avg_score_line, time_line]:
                f.write(line + '\n')
            print(title.strip())
            print(avg_score_line)
            print(time_line)
            print("-" * 50)
            # Add row for summary CSV
            summary_rows.append([
                model_name,
                penalty_type,
                evaluation['Average Score'],
                round(elapsed_time, 2)
            ])

            write_qualitative_analysis(model_name, penalty_type, under_threshold_reviews)

    if last_evaluation:
        final_block = ["====== Overall Final Metrics ======",
            f"Precision: {last_evaluation['Precision']}%",
            f"Recall: {last_evaluation['Recall']}%",
            f"F1-Score: {last_evaluation['F1-Score']}%",
            f"True Positives (TP): {last_evaluation['TP']}",
            f"False Positives (FP): {last_evaluation['FP']}",
            f"False Negatives (FN): {last_evaluation['FN']}"
        ]
        for line in final_block:
            f.write(line + '\n')
            print(line)

with open(summary_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Model", "Penalty", "Avg Score", "Time (s)"])
    for row in summary_rows:
        writer.writerow(row)

# Write sorted qualitative rows to CSV
qualitative_rows.sort(key=lambda x: x[2])  # sort by Review ID
with open(qual_file, 'w', newline='', encoding='utf-8') as qf:
    writer = csv.writer(qf)
    writer.writerow(["Model", "Penalty", "Review ID", "Review Text", "Score", "Actual Quad(s)", "Predicted Quad(s)"])
    for row in qualitative_rows:
        writer.writerow(row)

print(f"✅ All evaluations completed. Results saved in '{output_file}', qualitative samples in '{qual_file}', and summary in '{summary_csv}'.")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ (debug) Drive mounted, checking file paths:
Human File: /content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv -> Exists: True
ChatGPT File: /content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv -> Exists: True
✅ (debug) Loaded HumanAnnotations.csv with 1000 rows and columns: ['Review ID', 'Review', 'Aspect', 'Category', 'Opinion', 'Sentiment']
✅ (debug) Loaded ChatGPTAnnotations.csv with 503 rows and columns: ['Review ID', 'Review', 'Aspect', 'Category', 'Opinion', 'Sentiment']
✅ (debug) Found 376 unique Review IDs


<ipython-input-23-cf5858361844>:62: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-23-cf5858361844>:63: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ (debug) Sample review ID: 1
✅ (debug) Sample actual quads: [['staff', 'human resources', 'friendly and supportive', 'positive'], ['place', 'location', 'nice and pleasant', 'positive'], ['money', 'value', 'worth', 'positive']]
✅ (debug) Sample predicted quads: [['place', 'general', 'nice and peasant', 'positive'], ['money', 'value', 'worth', 'positive'], ['staff', 'human resources', 'very friendly and supportive', 'positive']]
====== Embedding-based (MiniLM) | Penalty: Hard ======
Average Score: 38.01%
Elapsed Time: 6.36 seconds
--------------------------------------------------
====== Embedding-based (MiniLM) | Penalty: Soft ======
Average Score: 60.08%
Elapsed Time: 59.17 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Hard ======
Average Score: 42.88%
Elapsed Time: 10.81 seconds
--------------------------------------------------
====== Instruction-Tuned (E5) | Penalty: Soft ======
Average Score: 67.74%
Elapsed Time: 100.62 seconds

# Extinsive analysis for one review

In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50

MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    # Aspect (embedding sim)
    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])

    # Opinion (embedding sim)
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    # Category (exact match or soft penalty)
    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    # Sentiment (exact match or soft penalty)
    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    # Weighted contributions
    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    # DEBUG PRINTS
    print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
    print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
    print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
    print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
    print(f"    Final Score: {final_score:.2f}/100")

    return final_score

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    # Load CSVs
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    # Convert all string cells to lowercase
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    # Ensure Review IDs are strings
    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    # Get data for the review
    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    # Get review text
    if not human_rows.empty:
        review_text = human_rows['Review'].iloc[0]
    elif not chatgpt_rows.empty:
        review_text = chatgpt_rows['Review'].iloc[0]
    else:
        review_text = ""

    # Collect quads
    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]

    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Compare predicted quads against actual quads
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score:.2f}/100")
        else:
            print("No actual quads available for comparison.")

# Example usage: loop over all models and penalty types
if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            debug_review_comparison(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)



🚀 Loading model: Embedding-based (MiniLM)


<ipython-input-22-e838eaa2ba8b>:72: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-22-e838eaa2ba8b>:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Embedding-based (MiniLM) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']
    [Aspect]    'staff' vs 'place': 0.6647 → 16.62/25
    [Opinion]   'friendly and supportive' vs 'nice and peasant': 0.6762 → 16.91/25
    [Category]  'human resources' vs 'general': no match → 0.00/25
    [Sentiment

<ipython-input-22-e838eaa2ba8b>:72: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-22-e838eaa2ba8b>:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Embedding-based (MiniLM) | Penalty: Soft
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']
    [Aspect]    'staff' vs 'place': 0.6647 → 16.62/25
    [Opinion]   'friendly and supportive' vs 'nice and peasant': 0.6762 → 16.91/25
    [Category]  'human resources' vs 'general': no match → 12.50/25
    [Sentimen

...

    [Aspect]    'place' vs 'place': 1.0000 → 25.00/25
    [Opinion]   'nice and pleasant' vs 'nice and peasant': 0.9430 → 23.57/25
    [Category]  'location' vs 'general': no match → 12.50/25
    [Sentiment] 'positive' vs 'positive': match → 25.00/25
    Final Score: 86.07/100

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['money', 'value', 'worth', 'positive']
    [Aspect]    'money' vs 'place': 0.8904 → 22.26/25
    [Opinion]   'worth' vs 'nice and peasant': 0.8662 → 21.65/25
    [Category]  'value' vs 'general': no match → 12.50/25
    [Sentiment] 'positive' vs 'positive': match → 25.00/25
    Final Score: 81.42/100

--- Best Match Result ---
Best Actual Match : ['place', 'location', 'nice and pleasant', 'positive']
Best Score        : 86.07/100

--- Comparing Predicted Quad ---
Predicted         : ['money', 'value', 'worth', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and sup

<ipython-input-22-e838eaa2ba8b>:72: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-22-e838eaa2ba8b>:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: GTR-T5 | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']
    [Aspect]    'staff' vs 'place': 0.7851 → 19.63/25
    [Opinion]   'friendly and supportive' vs 'nice and peasant': 0.8019 → 20.05/25
    [Category]  'human resources' vs 'general': no match → 0.00/25
    [Sentiment] 'positive' vs 'p

<ipython-input-22-e838eaa2ba8b>:72: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-22-e838eaa2ba8b>:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: GTR-T5 | Penalty: Soft
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']
    [Aspect]    'staff' vs 'place': 0.7851 → 19.63/25
    [Opinion]   'friendly and supportive' vs 'nice and peasant': 0.8019 → 20.05/25
    [Category]  'human resources' vs 'general': no match → 12.50/25
    [Sentiment] 'positive' vs '

## Update with TP, FP, FN

In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.75  # Threshold to consider a match (out of 1)

MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    final_score = (
        aspect_similarity * 0.25 +
        opinion_similarity * 0.25 +
        category_score * 0.25 +
        sentiment_score * 0.25
    )
    return final_score  # [0,1]

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)

    TP = len(matched_actual_indices)
    FP = len(predicted_quads) - len(matched_predicted_indices)
    FN = len(actual_quads) - len(matched_actual_indices)

    return TP, FP, FN

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    TP, FP, FN = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    print(f"False Positives (FP): {FP}")
    print(f"False Negatives (FN): {FN}")

if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            debug_review_comparison(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)



🚀 Loading model: Embedding-based (MiniLM)


<ipython-input-6-2e52bc954953>:79: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-2e52bc954953>:80: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-2e52bc954953>:79: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Embedding-based (MiniLM) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
False Positives (FP): 1
False Negatives (FN): 1

🔍 Model: Embedding-based (MiniLM) | Penalty: Soft
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']


<ipython-input-6-2e52bc954953>:80: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== TP / FP / FN Counts =====
True Positives (TP): 3
False Positives (FP): 0
False Negatives (FN): 0

🚀 Loading model: Instruction-Tuned (E5)


...


===== TP / FP / FN Counts =====
True Positives (TP): 3
False Positives (FP): 0
False Negatives (FN): 0

🚀 Loading model: GTR-T5


<ipython-input-6-2e52bc954953>:79: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-2e52bc954953>:80: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: GTR-T5 | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
False Positives (FP): 1
False Negatives (FN): 1

🔍 Model: GTR-T5 | Penalty: Soft
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'gener

<ipython-input-6-2e52bc954953>:79: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-2e52bc954953>:80: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== TP / FP / FN Counts =====
True Positives (TP): 3
False Positives (FP): 0
False Negatives (FN): 0


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.75  # Threshold to consider a match (out of 1)

MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    final_score = (
        aspect_similarity * 0.25 +
        opinion_similarity * 0.25 +
        category_score * 0.25 +
        sentiment_score * 0.25
    )
    return final_score  # [0,1]

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    # Unmatched predicted → FP
    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    # Unmatched actual → FN
    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            debug_review_comparison(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)



🚀 Loading model: Embedding-based (MiniLM)


<ipython-input-7-884e6da44cb0>:93: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-7-884e6da44cb0>:94: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-7-884e6da44cb0>:93: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-7-884e6da44cb0>:94: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Embedding-based (MiniLM) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 1
  ❌ Unmatched Predicted: ['place', 'general', 'nice and peasant', 'positive']

False Negati

<ipython-input-7-884e6da44cb0>:93: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-7-884e6da44cb0>:94: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Instruction-Tuned (E5) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 1
  ❌ Unmatched Predicted: ['place', 'general', 'nice and peasant', 'positive']

False Negative

...


===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0

🚀 Loading model: GTR-T5


<ipython-input-7-884e6da44cb0>:93: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-7-884e6da44cb0>:94: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: GTR-T5 | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 1
  ❌ Unmatched Predicted: ['place', 'general', 'nice and peasant', 'positive']

False Negatives (FN): 1
  ❌ Mi

<ipython-input-7-884e6da44cb0>:93: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-7-884e6da44cb0>:94: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.75  # Threshold to consider a match (out of 1)

MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    final_score = (
        aspect_similarity * 0.25 +
        opinion_similarity * 0.25 +
        category_score * 0.25 +
        sentiment_score * 0.25
    )
    return final_score  # [0,1]

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

    # Print formulas and results
    print("\n===== Metrics Calculation =====")
    print(f"Precision = TP / (TP + FP) = {TP} / ({TP} + {FP})")
    print(f"  → Precision: {precision:.4f}")
    print(f"Recall = TP / (TP + FN) = {TP} / ({TP} + {FN})")
    print(f"  → Recall: {recall:.4f}")
    print(f"F1 Score = 2 * Precision * Recall / (Precision + Recall)")
    print(f"        = 2 * {precision:.4f} * {recall:.4f} / ({precision:.4f} + {recall:.4f})")
    print(f"  → F1 Score: {f1:.4f}")

if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            debug_review_comparison(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)



🚀 Loading model: Embedding-based (MiniLM)


<ipython-input-8-5c8c6369c7f5>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-8-5c8c6369c7f5>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-8-5c8c6369c7f5>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Embedding-based (MiniLM) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 1
  ❌ Unmatched Predicted: ['place', 'general', 'nice and peasant', 'positive']

False Negati

<ipython-input-8-5c8c6369c7f5>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0

===== Metrics Calculation =====
Precision = TP / (TP + FP) = 3 / (3 + 0)
  → Precision: 1.0000
Recall = TP / (TP + FN) = 3 / (3 + 0)
  → Recall: 1.0000
F1 Score = 2 * Precision * Recall / (Precision + Recall)
        = 2 * 1.0000 * 1.0000 / (1.0000 + 1.0000)
  → F1 Score: 1.0000

🚀 Loading model: Instruction-Tuned (E5)


...


===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0

===== Metrics Calculation =====
Precision = TP / (TP + FP) = 3 / (3 + 0)
  → Precision: 1.0000
Recall = TP / (TP + FN) = 3 / (3 + 0)
  → Recall: 1.0000
F1 Score = 2 * Precision * Recall / (Precision + Recall)
        = 2 * 1.0000 * 1.0000 / (1.0000 + 1.0000)
  → F1 Score: 1.0000

🚀 Loading model: GTR-T5


<ipython-input-8-5c8c6369c7f5>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-8-5c8c6369c7f5>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: GTR-T5 | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 1
  ❌ Unmatched Predicted: ['place', 'general', 'nice and peasant', 'positive']

False Negatives (FN): 1
  ❌ Mi

<ipython-input-8-5c8c6369c7f5>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-8-5c8c6369c7f5>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0

===== Metrics Calculation =====
Precision = TP / (TP + FP) = 3 / (3 + 0)
  → Precision: 1.0000
Recall = TP / (TP + FN) = 3 / (3 + 0)
  → Recall: 1.0000
F1 Score = 2 * Precision * Recall / (Precision + Recall)
        = 2 * 1.0000 * 1.0000 / (1.0000 + 1.0000)
  → F1 Score: 1.0000


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.75  # Threshold to consider a match (out of 1)

MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    final_score = (
        aspect_similarity * 0.25 +
        opinion_similarity * 0.25 +
        category_score * 0.25 +
        sentiment_score * 0.25
    )
    return final_score  # [0,1]

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

    # Print formulas and results
    print("\n===== Metrics Calculation =====")
    print(f"Precision (%) = (TP / (TP + FP)) * 100 = ({TP} / ({TP} + {FP})) * 100")
    print(f"  → Precision: {precision:.2f}%")
    print(f"Recall (%) = (TP / (TP + FN)) * 100 = ({TP} / ({TP} + {FN})) * 100")
    print(f"  → Recall: {recall:.2f}%")
    print(f"F1 Score (%) = (2 * Precision * Recall) / (Precision + Recall)")
    print(f"             = (2 * {precision:.2f} * {recall:.2f}) / ({precision:.2f} + {recall:.2f})")
    print(f"  → F1 Score: {f1:.2f}%")

if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            debug_review_comparison(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)



🚀 Loading model: Embedding-based (MiniLM)


<ipython-input-4-a86a3cf5046e>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-4-a86a3cf5046e>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Embedding-based (MiniLM) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 1
  ❌ Unmatched Predicted: ['place', 'general', 'nice and peasant', 'positive']

False Negati

<ipython-input-4-a86a3cf5046e>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-4-a86a3cf5046e>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

<ipython-input-4-a86a3cf5046e>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-4-a86a3cf5046e>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: GTR-T5 | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== TP / FP / FN Counts =====
True Positives (TP): 2
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 1
  ❌ Unmatched Predicted: ['place', 'general', 'nice and peasant', 'positive']

False Negatives (FN): 1
  ❌ Mi

<ipython-input-4-a86a3cf5046e>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-4-a86a3cf5046e>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0

===== Metrics Calculation =====
Precision (%) = (TP / (TP + FP)) * 100 = (3 / (3 + 0)) * 100
  → Precision: 100.00%
Recall (%) = (TP / (TP + FN)) * 100 = (3 / (3 + 0)) * 100
  → Recall: 100.00%
F1 Score (%) = (2 * Precision * Recall) / (Precision + Recall)
             = (2 * 100.00 * 100.00) / (100.00 + 100.00)
  → F1 Score: 100.00%


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.75  # Threshold to consider a match (out of 1)

MODEL_OPTIONS = {
    "1": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "2": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
    "3": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
}
PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    final_score = (
        aspect_similarity * 0.25 +
        opinion_similarity * 0.25 +
        category_score * 0.25 +
        sentiment_score * 0.25
    )
    return final_score  # [0,1]

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

    # Print formulas and results
    print("\n===== Metrics Calculation =====")
    print(f"Precision (%) = (TP / (TP + FP)) * 100 = ({TP} / ({TP} + {FP})) * 100")
    print(f"  → Precision: {precision:.2f}%")
    print(f"Recall (%) = (TP / (TP + FN)) * 100 = ({TP} / ({TP} + {FN})) * 100")
    print(f"  → Recall: {recall:.2f}%")
    print(f"F1 Score (%) = (2 * Precision * Recall) / (Precision + Recall)")
    print(f"             = (2 * {precision:.2f} * {recall:.2f}) / ({precision:.2f} + {recall:.2f})")
    print(f"  → F1 Score: {f1:.2f}%")

if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            debug_review_comparison(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)



🚀 Loading model: Embedding-based (MiniLM)


<ipython-input-5-25a943c93123>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-25a943c93123>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Embedding-based (MiniLM) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['place', 'location', 'nice and pleasant', 'positive']

--- Comparing Predicted Quad ---
Pr

<ipython-input-5-25a943c93123>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-25a943c93123>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0

===== Metrics Calculation =====
Precision (%) = (TP / (TP + FP)) * 100 = (3 / (3 + 0)) * 100
  → Precision: 100.00%
Recall (%) = (TP / (TP + FN)) * 100 = (3 / (3 + 0)) * 100
  → Recall: 100.00%
F1 Score (%) = (2 * Precision * Recall) / (Precision + Recall)
             = (2 * 100.00 * 100.00) / (100.00 + 100.00)
  → F1 Score: 100.00%

🚀 Loading model: Instruction-Tuned (E5)


...


--- Comparing Predicted Quad ---
Predicted         : ['money', 'value', 'worth', 'positive']
Against Actual    : ['money', 'value', 'worth', 'positive']

--- Best Match Result ---
Best Actual Match : ['money', 'value', 'worth', 'positive']
Best Score        : 100.00/100

--- Comparing Predicted Quad ---
Predicted         : ['staff', 'human resources', 'very friendly and supportive', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']

--- Comparing Predicted Quad ---
Predicted         : ['staff', 'human resources', 'very friendly and supportive', 'positive']
Against Actual    : ['place', 'location', 'nice and pleasant', 'positive']

--- Comparing Predicted Quad ---
Predicted         : ['staff', 'human resources', 'very friendly and supportive', 'positive']
Against Actual    : ['money', 'value', 'worth', 'positive']

--- Best Match Result ---
Best Actual Match : ['staff', 'human resources', 'friendly and supportive', 'positive']
Best Scor

<ipython-input-5-25a943c93123>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-25a943c93123>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: GTR-T5 | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['place', 'location', 'nice and pleasant', 'positive']

--- Comparing Predicted Quad ---
Predicted         : 

<ipython-input-5-25a943c93123>:97: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-25a943c93123>:98: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



--- Comparing Predicted Quad ---
Predicted         : ['money', 'value', 'worth', 'positive']
Against Actual    : ['place', 'location', 'nice and pleasant', 'positive']

--- Comparing Predicted Quad ---
Predicted         : ['money', 'value', 'worth', 'positive']
Against Actual    : ['money', 'value', 'worth', 'positive']

--- Best Match Result ---
Best Actual Match : ['money', 'value', 'worth', 'positive']
Best Score        : 100.00/100

--- Comparing Predicted Quad ---
Predicted         : ['staff', 'human resources', 'very friendly and supportive', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']

--- Comparing Predicted Quad ---
Predicted         : ['staff', 'human resources', 'very friendly and supportive', 'positive']
Against Actual    : ['place', 'location', 'nice and pleasant', 'positive']

--- Comparing Predicted Quad ---
Predicted         : ['staff', 'human resources', 'very friendly and supportive', 'positive']
Against Actual 

# Final approach

In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.80  # Threshold to consider a match (out of 1)

MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}



PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    if verbose:
        print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
        print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
        print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
        print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
        print(f"    Final Score: {final_score:.2f}/100")

    return final_score / 100  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison with element scores
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

    # Print formulas and results
    print("\n===== Metrics Calculation =====")
    print(f"Precision (%) = (TP / (TP + FP)) * 100 = ({TP} / ({TP} + {FP})) * 100")
    print(f"  → Precision: {precision:.2f}%")
    print(f"Recall (%) = (TP / (TP + FN)) * 100 = ({TP} / ({TP} + {FN})) * 100")
    print(f"  → Recall: {recall:.2f}%")
    print(f"F1 Score (%) = (2 * Precision * Recall) / (Precision + Recall)")
    print(f"             = (2 * {precision:.2f} * {recall:.2f}) / ({precision:.2f} + {recall:.2f})")
    print(f"  → F1 Score: {f1:.2f}%")

if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            debug_review_comparison(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)




🚀 Loading model: Sentence-T5


<ipython-input-5-5ab2c0925f41>:111: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-5ab2c0925f41>:112: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Sentence-T5 | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']
    [Aspect]    'staff' vs 'place': 0.9081 → 22.70/25
    [Opinion]   'friendly and supportive' vs 'nice and peasant': 0.8991 → 22.48/25
    [Category]  'human resources' vs 'general': no match → 0.00/25
    [Sentiment] 'positive' 

<ipython-input-5-5ab2c0925f41>:111: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-5ab2c0925f41>:112: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


    [Aspect]    'staff' vs 'money': 0.9105 → 22.76/25
    [Opinion]   'friendly and supportive' vs 'worth': 0.8790 → 21.98/25
    [Category]  'human resources' vs 'value': no match → 12.50/25
    [Sentiment] 'positive' vs 'positive': match → 25.00/25
    Final Score: 82.24/100

--- Comparing Predicted Quad ---
Predicted         : ['money', 'value', 'worth', 'positive']
Against Actual    : ['place', 'location', 'nice and pleasant', 'positive']
    [Aspect]    'place' vs 'money': 0.9133 → 22.83/25
    [Opinion]   'nice and pleasant' vs 'worth': 0.8805 → 22.01/25
    [Category]  'location' vs 'value': no match → 12.50/25
    [Sentiment] 'positive' vs 'positive': match → 25.00/25
    Final Score: 82.34/100

--- Comparing Predicted Quad ---
Predicted         : ['money', 'value', 'worth', 'positive']
Against Actual    : ['money', 'value', 'worth', 'positive']
    [Aspect]    'money' vs 'money': 1.0000 → 25.00/25
    [Opinion]   'worth' vs 'worth': 1.0000 → 25.00/25
    [Category]  'value' vs

...

    [Aspect]    'money' vs 'staff': 0.6470 → 16.18/25
    [Opinion]   'worth' vs 'very friendly and supportive': 0.6158 → 15.39/25
    [Category]  'value' vs 'human resources': no match → 12.50/25
    [Sentiment] 'positive' vs 'positive': match → 25.00/25
    Final Score: 69.07/100

--- Best Match Result ---
Best Actual Match : ['staff', 'human resources', 'friendly and supportive', 'positive']
Best Score        : 99.04/100

===== TP / FP / FN Counts =====
True Positives (TP): 3
  ✅ Matched: Actual ['place', 'location', 'nice and pleasant', 'positive'] ↔ Predicted ['place', 'general', 'nice and peasant', 'positive']
  ✅ Matched: Actual ['money', 'value', 'worth', 'positive'] ↔ Predicted ['money', 'value', 'worth', 'positive']
  ✅ Matched: Actual ['staff', 'human resources', 'friendly and supportive', 'positive'] ↔ Predicted ['staff', 'human resources', 'very friendly and supportive', 'positive']

False Positives (FP): 0

False Negatives (FN): 0

===== Metrics Calculation =====
Precisio

<ipython-input-5-5ab2c0925f41>:111: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-5ab2c0925f41>:112: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



🔍 Model: Instruction-Tuned (E5) | Penalty: Hard
Review ID: 1
Review Text: the place is nice and peasant. worth for the money. staff was very friendly and supportive.

Actual Quads:
  ['staff', 'human resources', 'friendly and supportive', 'positive']
  ['place', 'location', 'nice and pleasant', 'positive']
  ['money', 'value', 'worth', 'positive']

Predicted Quads:
  ['place', 'general', 'nice and peasant', 'positive']
  ['money', 'value', 'worth', 'positive']
  ['staff', 'human resources', 'very friendly and supportive', 'positive']

===== Detailed Best-Match Comparison =====

--- Comparing Predicted Quad ---
Predicted         : ['place', 'general', 'nice and peasant', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']
    [Aspect]    'staff' vs 'place': 0.8831 → 22.08/25
    [Opinion]   'friendly and supportive' vs 'nice and peasant': 0.9104 → 22.76/25
    [Category]  'human resources' vs 'general': no match → 0.00/25
    [Sentiment] 

<ipython-input-5-5ab2c0925f41>:111: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-5ab2c0925f41>:112: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


    [Aspect]    'place' vs 'money': 0.8904 → 22.26/25
    [Opinion]   'nice and pleasant' vs 'worth': 0.8548 → 21.37/25
    [Category]  'location' vs 'value': no match → 12.50/25
    [Sentiment] 'positive' vs 'positive': match → 25.00/25
    Final Score: 81.13/100

--- Comparing Predicted Quad ---
Predicted         : ['money', 'value', 'worth', 'positive']
Against Actual    : ['money', 'value', 'worth', 'positive']
    [Aspect]    'money' vs 'money': 1.0000 → 25.00/25
    [Opinion]   'worth' vs 'worth': 1.0000 → 25.00/25
    [Category]  'value' vs 'value': match → 25.00/25
    [Sentiment] 'positive' vs 'positive': match → 25.00/25
    Final Score: 100.00/100

--- Best Match Result ---
Best Actual Match : ['money', 'value', 'worth', 'positive']
Best Score        : 100.00/100

--- Comparing Predicted Quad ---
Predicted         : ['staff', 'human resources', 'very friendly and supportive', 'positive']
Against Actual    : ['staff', 'human resources', 'friendly and supportive', 'positive']


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.75  # Threshold to consider a match (out of 1)


MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}



PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0, ""

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    explanation = (f"[Aspect] '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25\n"
                   f"[Opinion] '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25\n"
                   f"[Category] '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25\n"
                   f"[Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25\n"
                   f"Final Score: {final_score:.2f}/100\n")

    return final_score / 100, explanation  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score, _ = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

# Collect results
export_data = []

def process_review(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    detail_logs = []
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        best_explanation = ""
        for act_quad in actual_quads:
            score, explanation = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
                best_explanation = explanation
        detail_logs.append(f"Predicted: {pred_quad}\nBest Match: {best_actual}\n{best_explanation}Best Score: {best_score * 100:.2f}/100")

    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

    export_data.append({
        "Model Name": model_name,
        "Penalty Type": penalty_type,
        "Review ID": review_id,
        "Review Text": review_text,
        "Actual Quads": str(actual_quads),
        "Predicted Quads": str(predicted_quads),
        "Detailed Best-Match Comparison": "\n\n".join(detail_logs),
        "TP Count": TP,
        "FP Count": FP,
        "FN Count": FN,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "F1 Score (%)": f1
    })

if __name__ == "__main__":
    review_id_to_check = "1"

    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            process_review(review_id_to_check, model, model_name, penalty_type, human_file_path, chatgpt_file_path)

    # Export to Excel
    df = pd.DataFrame(export_data)
    df.to_excel("/content/drive/MyDrive/Project6/Experiments/ChatGPT/model_evaluation_export.xlsx", index=False)
    print("✅ Export completed: /content/drive/MyDrive/Project6/Experiments/ChatGPT/model_evaluation_export.xlsx")


<ipython-input-6-b54d07acdabf>:114: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-b54d07acdabf>:115: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-b54d07acdabf>:114: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-b54d07acdabf>:115: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-b54d07acdabf>:114: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else 

✅ Export completed: /content/drive/MyDrive/Project6/Experiments/ChatGPT/model_evaluation_export.xlsx


# Final Final Approach
## 15/05/2025


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import datetime
import csv
import time

from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.80  # Threshold to consider a match (out of 1)

MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}



PENALTY_OPTIONS = ["Hard", "Soft"]

def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    if verbose:
        print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
        print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
        print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
        print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
        print(f"    Final Score: {final_score:.2f}/100")

    return final_score / 100  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison with element scores
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

def calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=False):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    total_TP = total_FP = total_FN = 0
    all_review_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
        total_TP += TP
        total_FP += FP
        total_FN += FN

    precision, recall, f1 = calculate_precision_recall_f1(total_TP, total_FP, total_FN)

    print("\n===== Overall Evaluation Across All Reviews =====")
    print(f"Model: {model._first_module().__class__.__name__} | Penalty: {penalty_type}")
    print(f"Total TP: {total_TP}, Total FP: {total_FP}, Total FN: {total_FN}")
    print(f"Overall Precision: {precision:.2f}%")
    print(f"Overall Recall: {recall:.2f}%")
    print(f"Overall F1 Score: {f1:.2f}%")

    if return_metrics:
        return total_TP, total_FP, total_FN, precision, recall, f1


def export_results_to_csv(total_TP, total_FP, total_FN, precision, recall, f1, model_name, penalty_type):
    # Create timestamped folder name
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)

    file_path = os.path.join(export_dir, "evaluation_results.csv")

    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)"])
        writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}"])

    print(f"\n📁 Results exported to: {file_path}")

if __name__ == "__main__":
    # Create once for this run
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)
    file_path = os.path.join(export_dir, "evaluation_results.csv")

    # Write header once
    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)", "Time (seconds)"])

    # Loop through all models and penalties
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            print(f"\n🧪 Evaluating with Penalty: {penalty_type}")
            start_time = time.time()

            total_TP, total_FP, total_FN, precision, recall, f1 = calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=True)

            elapsed_time = time.time() - start_time

            # Append result
            with open(file_path, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}", f"{elapsed_time:.2f}"])

    print(f"\n✅ All evaluations completed. Results saved to: {file_path}")



# Final Final Approach
## 18/05/2025


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import datetime
import csv
import time

from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.80  # Threshold to consider a match (out of 1)
PENALTY_OPTIONS = ["Hard", "Soft"]

MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}




def compute_similarity_qual(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity_qual(model, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity_qual(model, actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads_qual(model, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity_qual(model, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews_qual(model, review_dict, threshold=MATCH_THRESHOLD):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads_qual(model, actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews


def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    if verbose:
        print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
        print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
        print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
        print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
        print(f"    Final Score: {final_score:.2f}/100")

    return final_score / 100  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison with element scores
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

def calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=False):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    total_TP = total_FP = total_FN = 0
    all_review_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
        total_TP += TP
        total_FP += FP
        total_FN += FN

    precision, recall, f1 = calculate_precision_recall_f1(total_TP, total_FP, total_FN)

    print("\n===== Overall Evaluation Across All Reviews =====")
    print(f"Model: {model._first_module().__class__.__name__} | Penalty: {penalty_type}")
    print(f"Total TP: {total_TP}, Total FP: {total_FP}, Total FN: {total_FN}")
    print(f"Overall Precision: {precision:.2f}%")
    print(f"Overall Recall: {recall:.2f}%")
    print(f"Overall F1 Score: {f1:.2f}%")

    if return_metrics:
        return total_TP, total_FP, total_FN, precision, recall, f1


def export_results_to_csv(total_TP, total_FP, total_FN, precision, recall, f1, model_name, penalty_type):
    # Create timestamped folder name
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)

    file_path = os.path.join(export_dir, "evaluation_results.csv")

    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)"])
        writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}"])

    print(f"\n📁 Results exported to: {file_path}")

if __name__ == "__main__":
    # Create once for this run
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)
    file_path = os.path.join(export_dir, "evaluation_results.csv")

    # Write header once
    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)", "Time (seconds)"])

    # Loop through all models and penalties
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            print(f"\n🧪 Evaluating with Penalty: {penalty_type}")
            start_time = time.time()

            total_TP, total_FP, total_FN, precision, recall, f1 = calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=True)

            elapsed_time = time.time() - start_time

            # Append result
            with open(file_path, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}", f"{elapsed_time:.2f}"])

    print(f"\n✅ All evaluations completed. Results saved to: {file_path}")




🚀 Loading model: Sentence-T5

🧪 Evaluating with Penalty: Hard


<ipython-input-5-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Hard
Total TP: 231, Total FP: 272, Total FN: 797
Overall Precision: 45.92%
Overall Recall: 22.47%
Overall F1 Score: 30.18%

🧪 Evaluating with Penalty: Soft


<ipython-input-5-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 486, Total FP: 17, Total FN: 591
Overall Precision: 96.62%
Overall Recall: 45.13%
Overall F1 Score: 61.52%

🚀 Loading model: GTR-T5

🧪 Evaluating with Penalty: Hard


...


===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 345, Total FP: 158, Total FN: 675
Overall Precision: 68.59%
Overall Recall: 33.82%
Overall F1 Score: 45.31%

🚀 Loading model: Instruction-Tuned (E5)

🧪 Evaluating with Penalty: Hard


<ipython-input-5-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Hard
Total TP: 231, Total FP: 272, Total FN: 799
Overall Precision: 45.92%
Overall Recall: 22.43%
Overall F1 Score: 30.14%

🧪 Evaluating with Penalty: Soft


<ipython-input-5-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-5-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 486, Total FP: 17, Total FN: 593
Overall Precision: 96.62%
Overall Recall: 45.04%
Overall F1 Score: 61.44%

✅ All evaluations completed. Results saved to: /content/drive/MyDrive/Project6/Experiments/ChatGPT/032318052025/evaluation_results.csv


## Qualitative Analysis

In [ ]:
# Load the CSV files
human_df = pd.read_csv(human_file_path)
chatgpt_df = pd.read_csv(chatgpt_file_path)

# Standardize text and ensure Review ID is string
human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
human_df['Review ID'] = human_df['Review ID'].astype(str)
chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

# Construct review_dict from actual data
review_dict = {}
all_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

for review_id in all_ids:
    actual_quads = human_df[human_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    predicted_quads = chatgpt_df[chatgpt_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    review_dict[review_id] = {
        "actual": actual_quads,
        "predicted": predicted_quads
    }

# Run qualitative evaluation
review_scores, flagged_reviews = evaluate_reviews_qual(model, review_dict)

# Output results
print("Scores:", review_scores)

if flagged_reviews:
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
else:
    print("\nNo reviews below threshold.")


<ipython-input-6-5fb66125bc5b>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
<ipython-input-6-5fb66125bc5b>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)


Scores: {'57': 95.03, '126': 93.9, '65': 0.0, '53': 84.53, '89': 91.62, '212': 0.0, '132': 92.2, '28': 93.03, '222': 0.0, '175': 0.0, '191': 0.0, '125': 88.36, '107': 89.03, '121': 91.52, '149': 0.0, '180': 0.0, '201': 0.0, '182': 0.0, '119': 0.0, '100': 94.97, '193': 0.0, '103': 95.65, '156': 0.0, '59': 89.92, '111': 89.16, '225': 0.0, '21': 91.16, '51': 91.08, '1': 97.36, '64': 93.5, '92': 97.85, '165': 0.0, '13': 80.57, '25': 89.7, '174': 0.0, '159': 0.0, '76': 99.26, '173': 0.0, '194': 0.0, '224': 0.0, '178': 0.0, '45': 90.09, '143': 88.3, '141': 88.05, '10': 0.0, '145': 0.0, '137': 92.91, '54': 83.72, '209': 0.0, '196': 0.0, '12': 100.0, '123': 86.23, '219': 0.0, '101': 85.82, '19': 94.09, '171': 0.0, '90': 79.37, '150': 0.0, '162': 0.0, '8': 92.36, '75': 98.36, '211': 0.0, '204': 0.0, '163': 0.0, '177': 0.0, '73': 87.21, '179': 0.0, '228': 0.0, '147': 0.0, '14': 0.0, '93': 85.94, '138': 86.96, '186': 0.0, '77': 92.04, '9': 86.35, '164': 0.0, '214': 0.0, '106': 88.65, '151': 0.0, 

# Final Final Final Approach
## 27/05/2025


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive



In [ ]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import datetime
import csv
import time

from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.80  # Threshold to consider a match (out of 1)
PENALTY_OPTIONS = ["Hard", "Soft"]

MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}




def compute_similarity_qual(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity_qual(model, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity_qual(model, actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads_qual(model, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity_qual(model, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews_qual(model, review_dict, threshold=MATCH_THRESHOLD):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads_qual(model, actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews


def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    if verbose:
        print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
        print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
        print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
        print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
        print(f"    Final Score: {final_score:.2f}/100")

    return final_score / 100  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison with element scores
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

def calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=False):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    total_TP = total_FP = total_FN = 0
    all_review_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
        total_TP += TP
        total_FP += FP
        total_FN += FN

    precision, recall, f1 = calculate_precision_recall_f1(total_TP, total_FP, total_FN)

    print("\n===== Overall Evaluation Across All Reviews =====")
    print(f"Model: {model._first_module().__class__.__name__} | Penalty: {penalty_type}")
    print(f"Total TP: {total_TP}, Total FP: {total_FP}, Total FN: {total_FN}")
    print(f"Overall Precision: {precision:.2f}%")
    print(f"Overall Recall: {recall:.2f}%")
    print(f"Overall F1 Score: {f1:.2f}%")

    if return_metrics:
        return total_TP, total_FP, total_FN, precision, recall, f1


def export_results_to_csv(total_TP, total_FP, total_FN, precision, recall, f1, model_name, penalty_type):
    # Create timestamped folder name
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)

    file_path = os.path.join(export_dir, "evaluation_results.csv")

    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)"])
        writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}"])

    print(f"\n📁 Results exported to: {file_path}")

if __name__ == "__main__":
    # Create once for this run
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)
    file_path = os.path.join(export_dir, "evaluation_results.csv")

    # Write header once
    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)", "Time (seconds)"])

    # Loop through all models and penalties
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            print(f"\n🧪 Evaluating with Penalty: {penalty_type}")
            start_time = time.time()

            total_TP, total_FP, total_FN, precision, recall, f1 = calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=True)

            elapsed_time = time.time() - start_time

            # Append result
            with open(file_path, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}", f"{elapsed_time:.2f}"])

    print(f"\n✅ All evaluations completed. Results saved to: {file_path}")




🚀 Loading model: Sentence-T5

🧪 Evaluating with Penalty: Hard


<ipython-input-6-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Hard
Total TP: 461, Total FP: 383, Total FN: 585
Overall Precision: 54.62%
Overall Recall: 44.07%
Overall F1 Score: 48.78%

🧪 Evaluating with Penalty: Soft


<ipython-input-6-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 822, Total FP: 22, Total FN: 303
Overall Precision: 97.39%
Overall Recall: 73.07%
Overall F1 Score: 83.49%

🚀 Loading model: GTR-T5


...


🧪 Evaluating with Penalty: Hard


<ipython-input-6-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Hard
Total TP: 461, Total FP: 383, Total FN: 589
Overall Precision: 54.62%
Overall Recall: 43.90%
Overall F1 Score: 48.68%

🧪 Evaluating with Penalty: Soft


<ipython-input-6-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 822, Total FP: 22, Total FN: 307
Overall Precision: 97.39%
Overall Recall: 72.81%
Overall F1 Score: 83.32%

✅ All evaluations completed. Results saved to: /content/drive/MyDrive/Project6/Experiments/ChatGPT/043029052025/evaluation_results.csv


## Qualitative Analysis

In [ ]:
# Load the CSV files
human_df = pd.read_csv(human_file_path)
chatgpt_df = pd.read_csv(chatgpt_file_path)

# Standardize text and ensure Review ID is string
human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
human_df['Review ID'] = human_df['Review ID'].astype(str)
chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

# Construct review_dict from actual data
review_dict = {}
all_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

for review_id in all_ids:
    actual_quads = human_df[human_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    predicted_quads = chatgpt_df[chatgpt_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    review_dict[review_id] = {
        "actual": actual_quads,
        "predicted": predicted_quads
    }

# Run qualitative evaluation
review_scores, flagged_reviews = evaluate_reviews_qual(model, review_dict)

# Output results
print("Scores:", review_scores)

if flagged_reviews:
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
else:
    print("\nNo reviews below threshold.")





<ipython-input-4-5fb66125bc5b>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
<ipython-input-4-5fb66125bc5b>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)


Scores: {'140': 89.9, '15': 0.0, '53': 87.87, '228': 96.32, '93': 90.94, '110': 90.08, '129': 96.18, '36': 89.53, '170': 0.0, '207': 97.09, '29': 84.68, '64': 95.16, '65': 0.0, '55': 0.0, '145': 97.95, '165': 93.9, '63': 93.89, '33': 92.82, '230': 94.23, '22': 0.0, '80': 93.84, '158': 94.9, '20': 95.27, '49': 95.08, '150': 90.93, '217': 96.25, '180': 89.51, '221': 0.0, '96': 0.0, '115': 100.0, '218': 96.49, '70': 89.83, '208': 93.58, '27': 94.29, '48': 0.0, '91': 90.37, '219': 95.08, '61': 92.47, '155': 0.0, '119': 0.0, '103': 97.09, '188': 98.31, '130': 0.0, '226': 89.6, '116': 94.39, '126': 95.19, '205': 86.36, '28': 94.3, '88': 94.47, '108': 95.47, '156': 86.81, '181': 89.58, '13': 0.0, '136': 87.09, '105': 100.0, '56': 91.31, '113': 90.32, '190': 96.05, '69': 90.99, '82': 90.31, '164': 83.6, '112': 94.45, '68': 93.27, '138': 89.69, '92': 97.92, '179': 90.16, '192': 90.18, '73': 89.01, '154': 93.51, '1': 97.59, '169': 0.0, '30': 88.85, '167': 92.97, '189': 86.01, '215': 93.56, '9': 

In [ ]:
import os
import csv
import datetime
import pandas as pd

# Load the CSV files
human_df = pd.read_csv(human_file_path)
chatgpt_df = pd.read_csv(chatgpt_file_path)

# Standardize text and ensure Review ID is string
human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
human_df['Review ID'] = human_df['Review ID'].astype(str)
chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

# Construct review_dict from actual data
review_dict = {}
all_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

for review_id in all_ids:
    actual_quads = human_df[human_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    predicted_quads = chatgpt_df[chatgpt_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    review_dict[review_id] = {
        "actual": actual_quads,
        "predicted": predicted_quads
    }

# Run qualitative evaluation
review_scores, flagged_reviews = evaluate_reviews_qual(model, review_dict)

# Output results
print("Scores:", review_scores)

if flagged_reviews:
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
else:
    print("\nNo reviews below threshold.")

# ===== Export to same folder =====
# Create timestamped folder
timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
os.makedirs(export_dir, exist_ok=True)

# Paths for output CSVs
results_path = os.path.join(export_dir, "qualitative_scores.csv")
flagged_path = os.path.join(export_dir, "flagged_reviews.csv")

# Save review scores
with open(results_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Review ID", "Score (%)"])
    for review_id, score in review_scores.items():
        if review_id != "overall_performance":
            writer.writerow([review_id, f"{score:.2f}"])
    writer.writerow(["Overall", f"{review_scores['overall_performance']:.2f}"])
print(f"📄 Review scores saved to: {results_path}")

# Save flagged reviews
if flagged_reviews:
    with open(flagged_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Review ID", "Actual Quads", "Predicted Quads", "Total Score"])
        for entry in flagged_reviews:
            writer.writerow([
                entry["review_id"],
                str(entry["actual"]),
                str(entry["predicted"]),
                f"{entry['total_score']:.2f}"
            ])
    print(f"⚠️ Flagged reviews saved to: {flagged_path}")
else:
    print("✅ No flagged reviews to export.")


<ipython-input-5-564a1f17b80b>:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
<ipython-input-5-564a1f17b80b>:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)


Scores: {'140': 89.9, '15': 0.0, '53': 87.87, '228': 96.32, '93': 90.94, '110': 90.08, '129': 96.18, '36': 89.53, '170': 0.0, '207': 97.09, '29': 84.68, '64': 95.16, '65': 0.0, '55': 0.0, '145': 97.95, '165': 93.9, '63': 93.89, '33': 92.82, '230': 94.23, '22': 0.0, '80': 93.84, '158': 94.9, '20': 95.27, '49': 95.08, '150': 90.93, '217': 96.25, '180': 89.51, '221': 0.0, '96': 0.0, '115': 100.0, '218': 96.49, '70': 89.83, '208': 93.58, '27': 94.29, '48': 0.0, '91': 90.37, '219': 95.08, '61': 92.47, '155': 0.0, '119': 0.0, '103': 97.09, '188': 98.31, '130': 0.0, '226': 89.6, '116': 94.39, '126': 95.19, '205': 86.36, '28': 94.3, '88': 94.47, '108': 95.47, '156': 86.81, '181': 89.58, '13': 0.0, '136': 87.09, '105': 100.0, '56': 91.31, '113': 90.32, '190': 96.05, '69': 90.99, '82': 90.31, '164': 83.6, '112': 94.45, '68': 93.27, '138': 89.69, '92': 97.92, '179': 90.16, '192': 90.18, '73': 89.01, '154': 93.51, '1': 97.59, '169': 0.0, '30': 88.85, '167': 92.97, '189': 86.01, '215': 93.56, '9': 

# Final Final Final Final Approach
## 29/05/2025


## **Attempt 1**


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive



In [ ]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import datetime
import csv
import time

from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.80  # Threshold to consider a match (out of 1)
PENALTY_OPTIONS = ["Hard", "Soft"]

MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}




def compute_similarity_qual(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity_qual(model, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity_qual(model, actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads_qual(model, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity_qual(model, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews_qual(model, review_dict, threshold=MATCH_THRESHOLD):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads_qual(model, actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews


def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    if verbose:
        print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
        print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
        print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
        print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
        print(f"    Final Score: {final_score:.2f}/100")

    return final_score / 100  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison with element scores
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

def calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=False):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    total_TP = total_FP = total_FN = 0
    all_review_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
        total_TP += TP
        total_FP += FP
        total_FN += FN

    precision, recall, f1 = calculate_precision_recall_f1(total_TP, total_FP, total_FN)

    print("\n===== Overall Evaluation Across All Reviews =====")
    print(f"Model: {model._first_module().__class__.__name__} | Penalty: {penalty_type}")
    print(f"Total TP: {total_TP}, Total FP: {total_FP}, Total FN: {total_FN}")
    print(f"Overall Precision: {precision:.2f}%")
    print(f"Overall Recall: {recall:.2f}%")
    print(f"Overall F1 Score: {f1:.2f}%")

    if return_metrics:
        return total_TP, total_FP, total_FN, precision, recall, f1


def export_results_to_csv(total_TP, total_FP, total_FN, precision, recall, f1, model_name, penalty_type):
    # Create timestamped folder name
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)

    file_path = os.path.join(export_dir, "evaluation_results.csv")

    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)"])
        writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}"])

    print(f"\n📁 Results exported to: {file_path}")

if __name__ == "__main__":
    # Create once for this run
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)
    file_path = os.path.join(export_dir, "evaluation_results.csv")

    # Write header once
    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)", "Time (seconds)"])

    # Loop through all models and penalties
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            print(f"\n🧪 Evaluating with Penalty: {penalty_type}")
            start_time = time.time()

            total_TP, total_FP, total_FN, precision, recall, f1 = calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=True)

            elapsed_time = time.time() - start_time

            # Append result
            with open(file_path, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}", f"{elapsed_time:.2f}"])

    print(f"\n✅ All evaluations completed. Results saved to: {file_path}")




🚀 Loading model: Sentence-T5

🧪 Evaluating with Penalty: Hard


<ipython-input-6-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Hard
Total TP: 461, Total FP: 383, Total FN: 585
Overall Precision: 54.62%
Overall Recall: 44.07%
Overall F1 Score: 48.78%

🧪 Evaluating with Penalty: Soft


<ipython-input-6-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 822, Total FP: 22, Total FN: 303
Overall Precision: 97.39%
Overall Recall: 73.07%
Overall F1 Score: 83.49%

🚀 Loading model: GTR-T5


...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


🧪 Evaluating with Penalty: Hard


<ipython-input-6-d702e240d208>:254: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-6-d702e240d208>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


### Qualitative Analysis

In [ ]:
# Load the CSV files
human_df = pd.read_csv(human_file_path)
chatgpt_df = pd.read_csv(chatgpt_file_path)

# Standardize text and ensure Review ID is string
human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
human_df['Review ID'] = human_df['Review ID'].astype(str)
chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

# Construct review_dict from actual data
review_dict = {}
all_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

for review_id in all_ids:
    actual_quads = human_df[human_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    predicted_quads = chatgpt_df[chatgpt_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    review_dict[review_id] = {
        "actual": actual_quads,
        "predicted": predicted_quads
    }

# Run qualitative evaluation
review_scores, flagged_reviews = evaluate_reviews_qual(model, review_dict)

# Output results
print("Scores:", review_scores)

if flagged_reviews:
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
else:
    print("\nNo reviews below threshold.")





<ipython-input-4-5fb66125bc5b>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
<ipython-input-4-5fb66125bc5b>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)


Scores: {'140': 89.9, '15': 0.0, '53': 87.87, '228': 96.32, '93': 90.94, '110': 90.08, '129': 96.18, '36': 89.53, '170': 0.0, '207': 97.09, '29': 84.68, '64': 95.16, '65': 0.0, '55': 0.0, '145': 97.95, '165': 93.9, '63': 93.89, '33': 92.82, '230': 94.23, '22': 0.0, '80': 93.84, '158': 94.9, '20': 95.27, '49': 95.08, '150': 90.93, '217': 96.25, '180': 89.51, '221': 0.0, '96': 0.0, '115': 100.0, '218': 96.49, '70': 89.83, '208': 93.58, '27': 94.29, '48': 0.0, '91': 90.37, '219': 95.08, '61': 92.47, '155': 0.0, '119': 0.0, '103': 97.09, '188': 98.31, '130': 0.0, '226': 89.6, '116': 94.39, '126': 95.19, '205': 86.36, '28': 94.3, '88': 94.47, '108': 95.47, '156': 86.81, '181': 89.58, '13': 0.0, '136': 87.09, '105': 100.0, '56': 91.31, '113': 90.32, '190': 96.05, '69': 90.99, '82': 90.31, '164': 83.6, '112': 94.45, '68': 93.27, '138': 89.69, '92': 97.92, '179': 90.16, '192': 90.18, '73': 89.01, '154': 93.51, '1': 97.59, '169': 0.0, '30': 88.85, '167': 92.97, '189': 86.01, '215': 93.56, '9': 

## **Attempt 2**


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive



In [ ]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import datetime
import csv
import time
import random

from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.80  # Threshold to consider a match (out of 1)
PENALTY_OPTIONS = ["Hard", "Soft"]

MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5 ", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}




def compute_similarity_qual(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity_qual(model, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity_qual(model, actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads_qual(model, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity_qual(model, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews_qual(model, review_dict, threshold=MATCH_THRESHOLD):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads_qual(model, actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews


def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    if verbose:
        print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
        print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
        print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
        print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
        print(f"    Final Score: {final_score:.2f}/100")

    return final_score / 100  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison with element scores
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

def calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=False):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    total_TP = total_FP = total_FN = 0
    all_review_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
        total_TP += TP
        total_FP += FP
        total_FN += FN

    precision, recall, f1 = calculate_precision_recall_f1(total_TP, total_FP, total_FN)

    print("\n===== Overall Evaluation Across All Reviews =====")
    print(f"Model: {model._first_module().__class__.__name__} | Penalty: {penalty_type}")
    print(f"Total TP: {total_TP}, Total FP: {total_FP}, Total FN: {total_FN}")
    print(f"Overall Precision: {precision:.2f}%")
    print(f"Overall Recall: {recall:.2f}%")
    print(f"Overall F1 Score: {f1:.2f}%")

    if return_metrics:
        return total_TP, total_FP, total_FN, precision, recall, f1


def export_results_to_csv(total_TP, total_FP, total_FN, precision, recall, f1, model_name, penalty_type):
    # Create timestamped folder name
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)

    file_path = os.path.join(export_dir, "evaluation_results.csv")

    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)"])
        writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}"])

    print(f"\n📁 Results exported to: {file_path}")


def sample_midrange_matches(model, model_name, penalty_type, human_file_path, chatgpt_file_path, output_csv_path, threshold=MATCH_THRESHOLD):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    all_review_ids = list(set(human_df['Review ID']).union(set(chatgpt_df['Review ID'])))
    random.shuffle(all_review_ids)

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
             str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
             str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        if not predicted_quads or not actual_quads:
            continue

        scores = []
        explanation = []

        for pred in predicted_quads:
            best_score = 0
            best_actual = None
            best_detail = ""
            for act in actual_quads:
                score = quadruple_similarity_reweighted(model, penalty_type, act, pred, verbose=False)
                if score > best_score:
                    best_score = score
                    best_actual = act

                    # Add detailed explanation
                    detail_lines = []
                    aspect_sim = compute_similarity(model, act[0], pred[0])
                    opinion_sim = compute_similarity(model, act[2], pred[2])
                    category_match = act[1].strip().lower() == pred[1].strip().lower()
                    sentiment_match = act[3].strip().lower() == pred[3].strip().lower()

                    detail_lines.append(f"Aspect: '{act[0]}' vs '{pred[0]}' → {aspect_sim:.2f}")
                    detail_lines.append(f"Opinion: '{act[2]}' vs '{pred[2]}' → {opinion_sim:.2f}")
                    detail_lines.append(f"Category: '{act[1]}' vs '{pred[1]}' → {'match' if category_match else 'no match'}")
                    detail_lines.append(f"Sentiment: '{act[3]}' vs '{pred[3]}' → {'match' if sentiment_match else 'no match'}")

                    best_detail = " | ".join(detail_lines)

            scores.append(best_score)
            explanation.append({
                "Predicted": pred,
                "Matched Actual": best_actual,
                "Score": round(best_score * 100, 2),
                "Details": best_detail
            })

        if not scores:
            continue

        avg_score = round(sum(scores) / len(scores) * 100, 2)

        if 0 < avg_score < 100:
            TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
            precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

            detailed_info = "\n\n".join([
                f"Predicted Quad: {e['Predicted']}\nMatched Actual: {e['Matched Actual']}\nScore: {e['Score']}%\n{e['Details']}"
                for e in explanation
            ])

            with open(output_csv_path, mode='w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow([
                    "Review ID", "Review Text", "Model Name", "Penalty Type",
                    "Actual Quads", "Predicted Quads",
                    "Precision (%)", "Recall (%)", "F1 Score (%)",
                    "SAFE Similarity Score (%)", "Detailed Matching Explanation"
                ])
                writer.writerow([
                    review_id,
                    review_text,
                    model_name,
                    penalty_type,
                    actual_quads,
                    predicted_quads,
                    f"{precision:.2f}",
                    f"{recall:.2f}",
                    f"{f1:.2f}",
                    avg_score,
                    detailed_info
                ])

            print(f"\n✅ Sampled and exported: Review ID {review_id} with SAFE Score {avg_score}%")
            return

    print("\n⚠️ No matching review found in the given criteria (0 < score < 100)")



if __name__ == "__main__":
    # Create once for this run
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)
    file_path = os.path.join(export_dir, "evaluation_results.csv")

    # Write header once
    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)", "Time (seconds)"])

    # Loop through all models and penalties
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            print(f"\n🧪 Evaluating with Penalty: {penalty_type}")
            start_time = time.time()

            total_TP, total_FP, total_FN, precision, recall, f1 = calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=True)

            elapsed_time = time.time() - start_time

            # Append result
            with open(file_path, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}", f"{elapsed_time:.2f}"])


            # Call midrange sample function to save detailed match explanation
            sample_output_path = os.path.join(export_dir, f"sample_midrange_result_{model_name.replace(' ', '_')}_{penalty_type}.csv")
            sample_midrange_matches(
                model=model,
                model_name=model_name,
                penalty_type=penalty_type,
                human_file_path=human_file_path,
                chatgpt_file_path=chatgpt_file_path,
                output_csv_path=sample_output_path)

    print(f"\n✅ All evaluations completed. Results saved to: {file_path}")




🚀 Loading model: Sentence-T5

🧪 Evaluating with Penalty: Hard


<ipython-input-14-54e28df3428a>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:256: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Hard
Total TP: 461, Total FP: 383, Total FN: 585
Overall Precision: 54.62%
Overall Recall: 44.07%
Overall F1 Score: 48.78%


<ipython-input-14-54e28df3428a>:317: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:318: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



✅ Sampled and exported: Review ID 112 with SAFE Score 79.82%

🧪 Evaluating with Penalty: Soft


...


✅ Sampled and exported: Review ID 205 with SAFE Score 76.35%

🧪 Evaluating with Penalty: Soft


<ipython-input-14-54e28df3428a>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:256: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 822, Total FP: 22, Total FN: 307
Overall Precision: 97.39%
Overall Recall: 72.81%
Overall F1 Score: 83.32%


<ipython-input-14-54e28df3428a>:317: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:318: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



✅ Sampled and exported: Review ID 47 with SAFE Score 87.17%

✅ All evaluations completed. Results saved to: /content/drive/MyDrive/Project6/Experiments/ChatGPT/055929052025/evaluation_results.csv


### Qualitative Analysis

In [ ]:
# Load the CSV files
human_df = pd.read_csv(human_file_path)
chatgpt_df = pd.read_csv(chatgpt_file_path)

# Standardize text and ensure Review ID is string
human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
human_df['Review ID'] = human_df['Review ID'].astype(str)
chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

# Construct review_dict from actual data
review_dict = {}
all_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

for review_id in all_ids:
    actual_quads = human_df[human_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    predicted_quads = chatgpt_df[chatgpt_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    review_dict[review_id] = {
        "actual": actual_quads,
        "predicted": predicted_quads
    }

# Run qualitative evaluation
review_scores, flagged_reviews = evaluate_reviews_qual(model, review_dict)

# Output results
print("Scores:", review_scores)

if flagged_reviews:
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
else:
    print("\nNo reviews below threshold.")





<ipython-input-15-721de97311de>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
<ipython-input-15-721de97311de>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)


Scores: {'140': 87.78, '15': 0.0, '53': 84.53, '228': 95.31, '93': 85.94, '110': 89.95, '129': 94.47, '36': 86.58, '170': 0.0, '207': 96.72, '29': 82.8, '64': 93.5, '65': 0.0, '55': 0.0, '145': 97.48, '165': 93.61, '63': 92.84, '33': 90.88, '230': 93.89, '22': 0.0, '80': 91.62, '158': 94.88, '20': 93.65, '49': 94.81, '150': 89.66, '217': 95.83, '180': 85.78, '221': 0.0, '96': 0.0, '115': 100.0, '218': 97.32, '70': 87.75, '208': 92.07, '27': 93.72, '48': 0.0, '91': 87.32, '219': 94.45, '61': 91.54, '155': 0.0, '119': 0.0, '103': 95.65, '188': 97.51, '130': 0.0, '226': 87.89, '116': 94.01, '126': 93.9, '205': 85.02, '28': 93.03, '88': 92.86, '108': 95.09, '156': 83.78, '181': 87.98, '13': 0.0, '136': 83.35, '105': 100.0, '56': 90.76, '113': 89.51, '190': 94.89, '69': 88.9, '82': 88.8, '164': 81.6, '112': 92.53, '68': 91.64, '138': 86.96, '92': 97.85, '179': 88.18, '192': 86.2, '73': 87.21, '154': 92.49, '1': 97.36, '169': 0.0, '30': 86.61, '167': 91.06, '189': 84.8, '215': 92.33, '9': 86

# Final Final Final Final Final Approach
## 30/05/2025


In [ ]:
import csv
import os
import time
import math

from sentence_transformers import SentenceTransformer, util
from google.colab import drive



In [ ]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import datetime
import csv
import time
import random

from sentence_transformers import SentenceTransformer, util

# Configurations
human_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/HumanAnnotations.csv'
chatgpt_file_path = '/content/drive/MyDrive/Project6/Experiments/ChatGPT/ChatGPTAnnotations.csv'
SOFT_PENALTY_FACTOR = 0.50
MATCH_THRESHOLD = 0.80  # Threshold to consider a match (out of 1)
PENALTY_OPTIONS = ["Hard", "Soft"]

MODEL_OPTIONS = {
    "1": ("Sentence-T5", "sentence-transformers/sentence-t5-base"),
    "2": ("GTR-T5 ", "sentence-transformers/gtr-t5-base"),
    "3": ("MPNet (SBERT)", "all-mpnet-base-v2"),
    "4": ("SimCSE (BERT)", "princeton-nlp/sup-simcse-bert-base-uncased"),
    "5": ("Embedding-based (MiniLM)", "all-MiniLM-L6-v2"),
    "6": ("Instruction-Tuned (E5)", "intfloat/e5-base"),
}




def compute_similarity_qual(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    return util.cos_sim(x_emb, y_emb).item()

def quadruple_similarity_qual(model, actual_quad, predicted_quad):
    if not actual_quad or not predicted_quad:
        return 0.0
    scores = []
    for actual_element, predicted_element in zip(actual_quad, predicted_quad):
        sim = compute_similarity_qual(model, actual_element, predicted_element)
        scores.append(sim)
    return sum(scores) / len(scores) * 100

def match_quads_qual(model, actual_quads, predicted_quads):
    total_score = 0
    matched = 0
    for pred_quad in predicted_quads:
        best_score = 0
        for act_quad in actual_quads:
            score = quadruple_similarity_qual(model, act_quad, pred_quad)
            if score > best_score:
                best_score = score
        total_score += best_score
        matched += 1
    if matched == 0:
        return 0.0
    return total_score / matched

def evaluate_reviews_qual(model, review_dict, threshold=MATCH_THRESHOLD):
    results = {}
    total_score = 0
    total_reviews = 0
    under_threshold_reviews = []

    for review_id, quads in review_dict.items():
        actual = quads.get("actual", [])
        predicted = quads.get("predicted", [])
        score = match_quads_qual(model, actual, predicted)
        rounded_score = round(score, 2)

        results[review_id] = rounded_score
        total_score += score
        total_reviews += 1

        if rounded_score < threshold:
            under_threshold_reviews.append({
                "review_id": review_id,
                "actual": actual,
                "predicted": predicted,
                "total_score": rounded_score
            })

    overall_score = total_score / total_reviews if total_reviews > 0 else 0.0
    results["overall_performance"] = round(overall_score, 2)

    return results, under_threshold_reviews


def compute_similarity(model, x, y):
    if x.strip().lower() == y.strip().lower():
        return 1.0
    x_emb = model.encode(x, convert_to_tensor=True)
    y_emb = model.encode(y, convert_to_tensor=True)
    sim = util.cos_sim(x_emb, y_emb).item()
    scaled_sim = (sim + 1) / 2  # shift range to [0, 1]
    return scaled_sim

def quadruple_similarity_reweighted(model, penalty_type, actual_quad, predicted_quad, verbose=False):
    if not actual_quad or not predicted_quad:
        return 0.0

    aspect_similarity = compute_similarity(model, actual_quad[0], predicted_quad[0])
    opinion_similarity = compute_similarity(model, actual_quad[2], predicted_quad[2])

    category_match = actual_quad[1].strip().lower() == predicted_quad[1].strip().lower()
    if penalty_type == "Hard":
        category_score = 1.0 if category_match else 0.0
    else:
        category_score = 1.0 if category_match else SOFT_PENALTY_FACTOR

    sentiment_match = actual_quad[3].strip().lower() == predicted_quad[3].strip().lower()
    if penalty_type == "Hard":
        sentiment_score = 1.0 if sentiment_match else 0.0
    else:
        sentiment_score = 1.0 if sentiment_match else SOFT_PENALTY_FACTOR

    aspect_contrib = aspect_similarity * 25
    opinion_contrib = opinion_similarity * 25
    category_contrib = category_score * 25
    sentiment_contrib = sentiment_score * 25

    final_score = aspect_contrib + opinion_contrib + category_contrib + sentiment_contrib

    if verbose:
        print(f"    [Aspect]    '{actual_quad[0]}' vs '{predicted_quad[0]}': {aspect_similarity:.4f} → {aspect_contrib:.2f}/25")
        print(f"    [Opinion]   '{actual_quad[2]}' vs '{predicted_quad[2]}': {opinion_similarity:.4f} → {opinion_contrib:.2f}/25")
        print(f"    [Category]  '{actual_quad[1]}' vs '{predicted_quad[1]}': {'match' if category_match else 'no match'} → {category_contrib:.2f}/25")
        print(f"    [Sentiment] '{actual_quad[3]}' vs '{predicted_quad[3]}': {'match' if sentiment_match else 'no match'} → {sentiment_contrib:.2f}/25")
        print(f"    Final Score: {final_score:.2f}/100")

    return final_score / 100  # scaled to 0-1

def calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type, threshold=MATCH_THRESHOLD):
    matched_actual_indices = set()
    matched_predicted_indices = set()
    tp_pairs = []
    fp_quads = []
    fn_quads = []

    for p_idx, pred_quad in enumerate(predicted_quads):
        best_score = 0
        best_a_idx = None
        for a_idx, act_quad in enumerate(actual_quads):
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad)
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        if best_score >= threshold:
            matched_actual_indices.add(best_a_idx)
            matched_predicted_indices.add(p_idx)
            tp_pairs.append((actual_quads[best_a_idx], pred_quad))

    for p_idx, pred_quad in enumerate(predicted_quads):
        if p_idx not in matched_predicted_indices:
            fp_quads.append(pred_quad)

    for a_idx, act_quad in enumerate(actual_quads):
        if a_idx not in matched_actual_indices:
            fn_quads.append(act_quad)

    TP = len(tp_pairs)
    FP = len(fp_quads)
    FN = len(fn_quads)

    return TP, FP, FN, tp_pairs, fp_quads, fn_quads

def calculate_precision_recall_f1(TP, FP, FN):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision * 100, recall * 100, f1 * 100  # scaled to percentage

def debug_review_comparison(review_id, model, model_name, penalty_type, human_file_path, chatgpt_file_path):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)
    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    human_rows = human_df[human_df['Review ID'] == review_id]
    chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

    if human_rows.empty and chatgpt_rows.empty:
        print(f"⚠️ Review ID {review_id} not found in either file.")
        return

    review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

    actual_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in human_rows.iterrows()
    ]
    predicted_quads = [
        [str(row['Aspect']).strip(), str(row['Category']).strip(),
         str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
        for _, row in chatgpt_rows.iterrows()
    ]

    print(f"\n==============================")
    print(f"🔍 Model: {model_name} | Penalty: {penalty_type}")
    print(f"Review ID: {review_id}")
    print(f"Review Text: {review_text}")
    print("\nActual Quads:")
    for quad in actual_quads:
        print(f"  {quad}")
    print("\nPredicted Quads:")
    for quad in predicted_quads:
        print(f"  {quad}")

    # Detailed best-match comparison with element scores
    print("\n===== Detailed Best-Match Comparison =====")
    for pred_quad in predicted_quads:
        best_score = 0
        best_actual = None
        for act_quad in actual_quads:
            print(f"\n--- Comparing Predicted Quad ---")
            print(f"Predicted         : {pred_quad}")
            print(f"Against Actual    : {act_quad}")
            score = quadruple_similarity_reweighted(model, penalty_type, act_quad, pred_quad, verbose=True)
            if score > best_score:
                best_score = score
                best_actual = act_quad
        print(f"\n--- Best Match Result ---")
        if best_actual:
            print(f"Best Actual Match : {best_actual}")
            print(f"Best Score        : {best_score * 100:.2f}/100")
        else:
            print("No actual quads available for comparison.")

    # TP/FP/FN and metrics
    TP, FP, FN, tp_pairs, fp_quads, fn_quads = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)

    print(f"\n===== TP / FP / FN Counts =====")
    print(f"True Positives (TP): {TP}")
    for actual, predicted in tp_pairs:
        print(f"  ✅ Matched: Actual {actual} ↔ Predicted {predicted}")
    print(f"\nFalse Positives (FP): {FP}")
    for pred in fp_quads:
        print(f"  ❌ Unmatched Predicted: {pred}")
    print(f"\nFalse Negatives (FN): {FN}")
    for act in fn_quads:
        print(f"  ❌ Missed Actual: {act}")

    # Calculate precision, recall, f1 (scaled to 100)
    precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

def calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=False):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    total_TP = total_FP = total_FN = 0
    all_review_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
            str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
        total_TP += TP
        total_FP += FP
        total_FN += FN

    precision, recall, f1 = calculate_precision_recall_f1(total_TP, total_FP, total_FN)

    print("\n===== Overall Evaluation Across All Reviews =====")
    print(f"Model: {model._first_module().__class__.__name__} | Penalty: {penalty_type}")
    print(f"Total TP: {total_TP}, Total FP: {total_FP}, Total FN: {total_FN}")
    print(f"Overall Precision: {precision:.2f}%")
    print(f"Overall Recall: {recall:.2f}%")
    print(f"Overall F1 Score: {f1:.2f}%")

    if return_metrics:
        return total_TP, total_FP, total_FN, precision, recall, f1


def export_results_to_csv(total_TP, total_FP, total_FN, precision, recall, f1, model_name, penalty_type):
    # Create timestamped folder name
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)

    file_path = os.path.join(export_dir, "evaluation_results.csv")

    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)"])
        writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}"])

    print(f"\n📁 Results exported to: {file_path}")


def sample_midrange_matches(model, model_name, penalty_type, human_file_path, chatgpt_file_path, output_csv_path, threshold=MATCH_THRESHOLD):
    human_df = pd.read_csv(human_file_path)
    chatgpt_df = pd.read_csv(chatgpt_file_path)

    human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    human_df['Review ID'] = human_df['Review ID'].astype(str)
    chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

    all_review_ids = list(set(human_df['Review ID']).union(set(chatgpt_df['Review ID'])))
    random.shuffle(all_review_ids)

    for review_id in all_review_ids:
        human_rows = human_df[human_df['Review ID'] == review_id]
        chatgpt_rows = chatgpt_df[chatgpt_df['Review ID'] == review_id]

        review_text = human_rows['Review'].iloc[0] if not human_rows.empty else chatgpt_rows['Review'].iloc[0]

        actual_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
             str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in human_rows.iterrows()
        ]
        predicted_quads = [
            [str(row['Aspect']).strip(), str(row['Category']).strip(),
             str(row['Opinion']).strip(), str(row['Sentiment']).strip()]
            for _, row in chatgpt_rows.iterrows()
        ]

        if not predicted_quads or not actual_quads:
            continue

        scores = []
        explanation = []

        for pred in predicted_quads:
            best_score = 0
            best_actual = None
            best_detail = ""
            for act in actual_quads:
                score = quadruple_similarity_reweighted(model, penalty_type, act, pred, verbose=False)
                if score > best_score:
                    best_score = score
                    best_actual = act

                    # Add detailed explanation
                    detail_lines = []
                    aspect_sim = compute_similarity(model, act[0], pred[0])
                    opinion_sim = compute_similarity(model, act[2], pred[2])
                    category_match = act[1].strip().lower() == pred[1].strip().lower()
                    sentiment_match = act[3].strip().lower() == pred[3].strip().lower()

                    detail_lines.append(f"Aspect: '{act[0]}' vs '{pred[0]}' → {aspect_sim:.2f}")
                    detail_lines.append(f"Opinion: '{act[2]}' vs '{pred[2]}' → {opinion_sim:.2f}")
                    detail_lines.append(f"Category: '{act[1]}' vs '{pred[1]}' → {'match' if category_match else 'no match'}")
                    detail_lines.append(f"Sentiment: '{act[3]}' vs '{pred[3]}' → {'match' if sentiment_match else 'no match'}")

                    best_detail = " | ".join(detail_lines)

            scores.append(best_score)
            explanation.append({
                "Predicted": pred,
                "Matched Actual": best_actual,
                "Score": round(best_score * 100, 2),
                "Details": best_detail
            })

        if not scores:
            continue

        avg_score = round(sum(scores) / len(scores) * 100, 2)

        if 0 < avg_score < 100:
            TP, FP, FN, _, _, _ = calculate_tp_fp_fn(actual_quads, predicted_quads, model, penalty_type)
            precision, recall, f1 = calculate_precision_recall_f1(TP, FP, FN)

            detailed_info = "\n\n".join([
                f"Predicted Quad: {e['Predicted']}\nMatched Actual: {e['Matched Actual']}\nScore: {e['Score']}%\n{e['Details']}"
                for e in explanation
            ])

            with open(output_csv_path, mode='w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow([
                    "Review ID", "Review Text", "Model Name", "Penalty Type",
                    "Actual Quads", "Predicted Quads",
                    "Precision (%)", "Recall (%)", "F1 Score (%)",
                    "SAFE Similarity Score (%)", "Detailed Matching Explanation"
                ])
                writer.writerow([
                    review_id,
                    review_text,
                    model_name,
                    penalty_type,
                    actual_quads,
                    predicted_quads,
                    f"{precision:.2f}",
                    f"{recall:.2f}",
                    f"{f1:.2f}",
                    avg_score,
                    detailed_info
                ])

            print(f"\n✅ Sampled and exported: Review ID {review_id} with SAFE Score {avg_score}%")
            return

    print("\n⚠️ No matching review found in the given criteria (0 < score < 100)")



if __name__ == "__main__":
    # Create once for this run
    timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
    export_dir = f"/content/drive/MyDrive/Project6/Experiments/ChatGPT/{timestamp}"
    os.makedirs(export_dir, exist_ok=True)
    file_path = os.path.join(export_dir, "evaluation_results.csv")

    # Write header once
    with open(file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Model", "Penalty Type", "Total TP", "Total FP", "Total FN", "Precision (%)", "Recall (%)", "F1 Score (%)", "Time (seconds)"])

    # Loop through all models and penalties
    for model_id, (model_name, model_path) in MODEL_OPTIONS.items():
        print(f"\n🚀 Loading model: {model_name}")
        model = SentenceTransformer(model_path)
        for penalty_type in PENALTY_OPTIONS:
            print(f"\n🧪 Evaluating with Penalty: {penalty_type}")
            start_time = time.time()

            total_TP, total_FP, total_FN, precision, recall, f1 = calculate_total_metrics(human_file_path, chatgpt_file_path, model, penalty_type, return_metrics=True)

            elapsed_time = time.time() - start_time

            # Append result
            with open(file_path, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([model_name, penalty_type, total_TP, total_FP, total_FN, f"{precision:.2f}", f"{recall:.2f}", f"{f1:.2f}", f"{elapsed_time:.2f}"])


            # Call midrange sample function to save detailed match explanation
            sample_output_path = os.path.join(export_dir, f"sample_midrange_result_{model_name.replace(' ', '_')}_{penalty_type}.csv")
            sample_midrange_matches(
                model=model,
                model_name=model_name,
                penalty_type=penalty_type,
                human_file_path=human_file_path,
                chatgpt_file_path=chatgpt_file_path,
                output_csv_path=sample_output_path)

    print(f"\n✅ All evaluations completed. Results saved to: {file_path}")




🚀 Loading model: Sentence-T5

🧪 Evaluating with Penalty: Hard


<ipython-input-14-54e28df3428a>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:256: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Hard
Total TP: 461, Total FP: 383, Total FN: 585
Overall Precision: 54.62%
Overall Recall: 44.07%
Overall F1 Score: 48.78%


<ipython-input-14-54e28df3428a>:317: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:318: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



✅ Sampled and exported: Review ID 112 with SAFE Score 79.82%

🧪 Evaluating with Penalty: Soft


...


✅ Sampled and exported: Review ID 205 with SAFE Score 76.35%

🧪 Evaluating with Penalty: Soft


<ipython-input-14-54e28df3428a>:255: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:256: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



===== Overall Evaluation Across All Reviews =====
Model: Transformer | Penalty: Soft
Total TP: 822, Total FP: 22, Total FN: 307
Overall Precision: 97.39%
Overall Recall: 72.81%
Overall F1 Score: 83.32%


<ipython-input-14-54e28df3428a>:317: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-14-54e28df3428a>:318: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



✅ Sampled and exported: Review ID 47 with SAFE Score 87.17%

✅ All evaluations completed. Results saved to: /content/drive/MyDrive/Project6/Experiments/ChatGPT/055929052025/evaluation_results.csv


### Qualitative Analysis

In [ ]:
# Load the CSV files
human_df = pd.read_csv(human_file_path)
chatgpt_df = pd.read_csv(chatgpt_file_path)

# Standardize text and ensure Review ID is string
human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
human_df['Review ID'] = human_df['Review ID'].astype(str)
chatgpt_df['Review ID'] = chatgpt_df['Review ID'].astype(str)

# Construct review_dict from actual data
review_dict = {}
all_ids = set(human_df['Review ID']).union(set(chatgpt_df['Review ID']))

for review_id in all_ids:
    actual_quads = human_df[human_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    predicted_quads = chatgpt_df[chatgpt_df['Review ID'] == review_id][['Aspect', 'Category', 'Opinion', 'Sentiment']].dropna().values.tolist()
    review_dict[review_id] = {
        "actual": actual_quads,
        "predicted": predicted_quads
    }

# Run qualitative evaluation
review_scores, flagged_reviews = evaluate_reviews_qual(model, review_dict)

# Output results
print("Scores:", review_scores)

if flagged_reviews:
    print("\nReviews below threshold:")
    for entry in flagged_reviews:
        print(entry)
else:
    print("\nNo reviews below threshold.")





<ipython-input-15-721de97311de>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  human_df = human_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)
<ipython-input-15-721de97311de>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chatgpt_df = chatgpt_df.applymap(lambda x: x.lower().strip() if isinstance(x, str) else x)


Scores: {'140': 87.78, '15': 0.0, '53': 84.53, '228': 95.31, '93': 85.94, '110': 89.95, '129': 94.47, '36': 86.58, '170': 0.0, '207': 96.72, '29': 82.8, '64': 93.5, '65': 0.0, '55': 0.0, '145': 97.48, '165': 93.61, '63': 92.84, '33': 90.88, '230': 93.89, '22': 0.0, '80': 91.62, '158': 94.88, '20': 93.65, '49': 94.81, '150': 89.66, '217': 95.83, '180': 85.78, '221': 0.0, '96': 0.0, '115': 100.0, '218': 97.32, '70': 87.75, '208': 92.07, '27': 93.72, '48': 0.0, '91': 87.32, '219': 94.45, '61': 91.54, '155': 0.0, '119': 0.0, '103': 95.65, '188': 97.51, '130': 0.0, '226': 87.89, '116': 94.01, '126': 93.9, '205': 85.02, '28': 93.03, '88': 92.86, '108': 95.09, '156': 83.78, '181': 87.98, '13': 0.0, '136': 83.35, '105': 100.0, '56': 90.76, '113': 89.51, '190': 94.89, '69': 88.9, '82': 88.8, '164': 81.6, '112': 92.53, '68': 91.64, '138': 86.96, '92': 97.85, '179': 88.18, '192': 86.2, '73': 87.21, '154': 92.49, '1': 97.36, '169': 0.0, '30': 86.61, '167': 91.06, '189': 84.8, '215': 92.33, '9': 86